In [ ]:
%pip install numpy
%pip install pandas
%pip install matplotlib
%pip install seaborn
%pip install pathlib
%pip install scikit-learn
%pip install imbalanced-learn
%pip install scipy
%pip install xgboost

Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
    package_set, _ = create_package_set_from_installed()
              

In [1]:
import os
import multiprocessing

num_cores = multiprocessing.cpu_count()
print(f"La máquina tiene {num_cores} núcleos disponibles.")
hilos_optimos = str(min(num_cores, 64)) 
print(f"Configurando OpenBLAS para usar {hilos_optimos} hilos máximos...")
os.environ['OPENBLAS_NUM_THREADS'] = hilos_optimos
os.environ['MKL_NUM_THREADS'] = hilos_optimos
os.environ['OMP_NUM_THREADS'] = hilos_optimos

La máquina tiene 224 núcleos disponibles.
Configurando OpenBLAS para usar 64 hilos máximos...


In [2]:
import numpy as np
import pandas as pd
import itertools
import json
from pathlib import Path
from collections import Counter
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif, SelectKBest
from sklearn.svm import SVC
from sklearn.metrics import (f1_score, accuracy_score, recall_score,classification_report, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
from imblearn.combine import SMOTEENN


In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import xgboost as xgb
import numpy as np
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight
import gc

In [4]:
X = pd.read_pickle("Sets_Xy/X.pkl")
y = pd.read_pickle("Sets_Xy/y.pkl")

from sklearn.model_selection import train_test_split

#Division estratificada para muestras de cada clase a nivel de cada subset

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, 
    random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, 
    random_state=42)

#Encoding de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_val = le.transform(y_val)
y_test = le.transform(y_test)

mapeo_labels = pd.DataFrame({
    "label_original": le.classes_,
    "label_encoded": range(len(le.classes_))
})
print("Mapeo de etiquetas:\n", mapeo_labels)
class_names = le.classes_

Mapeo de etiquetas:
                label_original  label_encoded
0                      BENIGN              0
1                         Bot              1
2                        DDoS              2
3               DoS GoldenEye              3
4                    DoS Hulk              4
5            DoS Slowhttptest              5
6               DoS slowloris              6
7                 FTP-Patator              7
8                  Heartbleed              8
9                Infiltration              9
10                   PortScan             10
11                SSH-Patator             11
12    Web Attack  Brute Force             12
13  Web Attack  Sql Injection             13
14            Web Attack  XSS             14


In [5]:
import joblib

X_train_none_2 = joblib.load("Sets_Oversampling_2/X_train_none.joblib")
y_train_none_2 = joblib.load("Sets_Oversampling_2/y_train_none.joblib")

X_train_smote_2 = joblib.load("Sets_Oversampling_2/X_train_smote.joblib")
y_train_smote_2 = joblib.load("Sets_Oversampling_2/y_train_smote.joblib")

X_train_smote_enn_2 = joblib.load("Sets_Oversampling_2/X_train_smote_enn.joblib")
y_train_smote_enn_2 = joblib.load("Sets_Oversampling_2/y_train_smote_enn.joblib")

X_train_smote_tomek_2 = joblib.load("Sets_Oversampling_2/X_train_smote_tomek.joblib")
y_train_smote_tomek_2 = joblib.load("Sets_Oversampling_2/y_train_smote_tomek.joblib")

X_val = pd.read_pickle("Sets_Post_Scaled_Imp/X_val_scaled_bayesian.pkl")
y_val = pd.DataFrame(y_val)

X_test = pd.read_pickle("Sets_Post_Scaled_Imp/X_test_scaled_bayesian.pkl")
y_test = pd.DataFrame(y_test)


In [6]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import ExtraTreesClassifier

class FeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, method='tree', k=50, class_weight='balanced', random_state=42):
        self.method = method
        self.k = k
        self.class_weight = class_weight
        self.random_state = random_state
        self.selector = None
        self.feature_names_ = None
        self.support_mask_ = None

    def fit(self, X, y):
        self.feature_names_ = X.columns.tolist() if hasattr(X, 'columns') else list(range(X.shape[1]))
        
        X_array = X.values if hasattr(X, 'values') else X
        y_array = y.values if hasattr(y, 'values') else y
        
        total_features = X_array.shape[1]
        actual_k = min(self.k, total_features)
        
        if self.method == 'none':
            self.support_mask_ = np.ones(total_features, dtype=bool)
            return self

        if self.method == 'f_classif':
            self.selector = SelectKBest(score_func=f_classif, k=actual_k)
            self.selector.fit(X_array, y_array)
            self.support_mask_ = self.selector.get_support()

        elif self.method == 'tree':
            estimator = ExtraTreesClassifier(
                n_estimators=100,               
                max_depth=20,                   
                class_weight=self.class_weight,
                criterion='gini',               
                random_state=self.random_state, 
                n_jobs=-1
            )
            estimator.fit(X_array, y_array)
            importances = estimator.feature_importances_
            
            top_k_indices = np.argsort(importances)[-actual_k:]
            
            self.support_mask_ = np.zeros(total_features, dtype=bool)
            self.support_mask_[top_k_indices] = True

        return self

    def transform(self, X):
        if self.method == 'none':
            return X
            
        X_array = X.values if hasattr(X, 'values') else X
        X_transformed = X_array[:, self.support_mask_]

        if hasattr(X, 'columns'):
            selected_features = np.array(self.feature_names_)[self.support_mask_].tolist()
            return pd.DataFrame(X_transformed, columns=selected_features, index=X.index)
        else:
            return X_transformed

In [7]:
import numpy as np
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight

def balanced_class_weight(y):
    classes = np.unique(y)
    weights = compute_class_weight('balanced', classes=classes, y=y)
    return dict(zip(classes, weights))

def polynomial_class_weight(y, alpha=0.25):
    classes, counts = np.unique(y, return_counts=True)
    weights = 1.0 / np.power(counts, alpha)
    weights = weights / np.mean(weights)
    return dict(zip(classes, weights))

def effective_samples_class_weight(y, beta=0.999):
    classes, counts = np.unique(y, return_counts=True)
    effective_num = 1.0 - np.power(beta, counts)
    weights = (1.0 - beta) / effective_num
    weights = weights / np.mean(weights)
    return dict(zip(classes, weights))

def focal_class_weight_improved(y, gamma=2.0):

    classes, counts = np.unique(y, return_counts=True)
    probs = counts / np.sum(counts)
    weights = (1 - probs) ** gamma / probs
    weights = weights / np.mean(weights)
    return dict(zip(classes, weights))

In [8]:
import joblib
X_train_none_grouped = joblib.load("Sets_Oversampling_Grouped/X_train_none.joblib")
y_train_none_grouped = joblib.load("Sets_Oversampling_Grouped/y_train_none.joblib")

X_train_smote_grouped = joblib.load("Sets_Oversampling_Grouped/X_train_smote.joblib")
y_train_smote_grouped = joblib.load("Sets_Oversampling_Grouped/y_train_smote.joblib")

X_train_smote_enn_grouped = joblib.load("Sets_Oversampling_Grouped/X_train_smote_enn.joblib")
y_train_smote_enn_grouped = joblib.load("Sets_Oversampling_Grouped/y_train_smote_enn.joblib")

X_train_smote_tomek_grouped = joblib.load("Sets_Oversampling_Grouped/X_train_smote_tomek.joblib")
y_train_smote_tomek_grouped = joblib.load("Sets_Oversampling_Grouped/y_train_smote_tomek.joblib")

X_val_imp_grouped = pd.read_pickle("Sets_Post_Scaled_Imp/X_val_scaled_bayesian_grouped.pkl")
X_test_imp_grouped = pd.read_pickle("Sets_Post_Scaled_Imp/X_test_scaled_bayesian_grouped.pkl")

y_val_grouped = pd.read_pickle("Sets_Post_Scaled_Imp/y_val_grouped.pkl")
y_test_grouped = pd.read_pickle("Sets_Post_Scaled_Imp/y_test_grouped.pkl")

In [9]:
train_datasets_grouped = {
    'none': (X_train_none_grouped, y_train_none_grouped.values.ravel() if isinstance(y_train_none_grouped, pd.DataFrame) else y_train_none_grouped.ravel()),
    'smote': (X_train_smote_grouped, y_train_smote_grouped.values.ravel() if isinstance(y_train_smote_grouped, pd.DataFrame) else y_train_smote_grouped.ravel()),
    'smote_enn': (X_train_smote_enn_grouped, y_train_smote_enn_grouped.values.ravel() if isinstance(y_train_smote_enn_grouped, pd.DataFrame) else y_train_smote_enn_grouped.ravel()),
    'smote_tomek': (X_train_smote_tomek_grouped, y_train_smote_tomek_grouped.values.ravel() if isinstance(y_train_smote_tomek_grouped, pd.DataFrame) else y_train_smote_tomek_grouped.ravel())
}

In [10]:
train_datasets = {
    'none': (X_train_none_2, y_train_none_2.values.ravel() if isinstance(y_train_none_2, pd.DataFrame) else y_train_none_2.ravel()),
    'smote': (X_train_smote_2, y_train_smote_2.values.ravel() if isinstance(y_train_smote_2, pd.DataFrame) else y_train_smote_2.ravel()),
    'smote_enn': (X_train_smote_enn_2, y_train_smote_enn_2.values.ravel() if isinstance(y_train_smote_enn_2, pd.DataFrame) else y_train_smote_enn_2.ravel()),
    'smote_tomek': (X_train_smote_tomek_2, y_train_smote_tomek_2.values.ravel() if isinstance(y_train_smote_tomek_2, pd.DataFrame) else y_train_smote_tomek_2.ravel())
}

y_val_1d = y_val.values.ravel() if isinstance(y_val, pd.DataFrame) else y_val.ravel()
y_test_1d = y_test.values.ravel() if isinstance(y_test, pd.DataFrame) else y_test.ravel()

In [ ]:
import os
import gc
import copy
import warnings
import optuna
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from torch.optim.lr_scheduler import ReduceLROnPlateau

warnings.filterwarnings("ignore")

os.environ["CUDA_VISIBLE_DEVICES"] = "1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device_type = 'cuda' if torch.cuda.is_available() else 'cpu'

log_folder = "Logs_MLP_Baseline_1"
os.makedirs(log_folder, exist_ok=True)

def save_confusion_matrix_mlp(y_true, y_pred, trial_number, phase="Val"):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'CM MLP Baseline - Trial {trial_number} ({phase})')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig(f'{log_folder}/Trial_{trial_number}_MLP_CM.png')
    plt.close()

class TensorDataLoader:
    def __init__(self, X, y, batch_size, shuffle=False):
        self.X = X
        self.y = y
        self.dataset_len = self.X.shape[0]
        self.batch_size = batch_size
        self.shuffle = shuffle

    def __iter__(self):
        if self.shuffle:
            self.indices = torch.randperm(self.dataset_len, device=device)
        else:
            self.indices = None
        self.i = 0
        return self

    def __next__(self):
        if self.i >= self.dataset_len:
            raise StopIteration
        
        if self.indices is not None:
            idx = self.indices[self.i : self.i + self.batch_size]
            batch_X = self.X[idx]
            batch_y = self.y[idx]
        else:
            batch_X = self.X[self.i : self.i + self.batch_size]
            batch_y = self.y[self.i : self.i + self.batch_size]
            
        self.i += self.batch_size
        return batch_X, batch_y

    def __len__(self):
        return (self.dataset_len + self.batch_size - 1) // self.batch_size

class TabularMLP(nn.Module):
    def __init__(self, input_dim, hidden_layers, dropout_rate, activation_name, num_classes=15):
        super().__init__()
        activations = {
            'leaky_relu': nn.LeakyReLU(0.01),
            'gelu': nn.GELU(),
            'relu': nn.ReLU()
        }
        activation_layer = activations[activation_name]

        layers = []
        in_f = input_dim
        
        for out_f in hidden_layers:
            layers.append(nn.Linear(in_f, out_f))
            layers.append(nn.BatchNorm1d(out_f))
            layers.append(activation_layer)
            layers.append(nn.Dropout(dropout_rate))
            in_f = out_f
        
        layers.append(nn.Linear(in_f, num_classes))
        self.net = nn.Sequential(*layers)
        
    def forward(self, x): 
        return self.net(x)

print("Preparando tensores en VRAM...")
# NOTA: Asegúrate de que X_val, y_val_1d, train_datasets['none'] estén definidos en tu entorno
X_val_vram = torch.tensor(np.array(X_val, dtype=np.float32)).to(device)
y_val_vram = torch.tensor(np.array(y_val_1d, dtype=np.int64)).to(device)
y_val_cpu = y_val_vram.cpu().numpy()

x_raw, y_raw = train_datasets['none']
x_train_vram = torch.tensor(np.array(x_raw, dtype=np.float32)).to(device)
y_train_vram = torch.tensor(np.array(y_raw, dtype=np.int64)).to(device)

input_dimension = x_train_vram.shape[1]

def objective_mlp(trial):
    batch_size = trial.suggest_categorical('batch_size', [4096, 8192, 16384])
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)
    activation_name = trial.suggest_categorical('activation', ['leaky_relu', 'gelu', 'relu'])
    lr = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True)
    epochs = trial.suggest_int('epochs', 50, 100)
    
    n_layers = trial.suggest_int('n_layers', 2, 4)
    shape_strategy = trial.suggest_categorical('mlp_shape', ['bottleneck', 'flat'])
    
    hidden_layers = []
    base_units = trial.suggest_int('n_units_l0', 256, 1024, step=256)
    hidden_layers.append(base_units)
    
    if shape_strategy == 'flat':
        for _ in range(1, n_layers):
            hidden_layers.append(base_units)
    else:
        prev_units = base_units
        for i in range(1, n_layers):
            units = trial.suggest_int(f'n_units_l{i}', 64, prev_units, step=64)
            hidden_layers.append(units)
            prev_units = units
            
    # Dataloaders
    train_loader = TensorDataLoader(x_train_vram, y_train_vram, batch_size=batch_size, shuffle=True)
    val_loader = TensorDataLoader(X_val_vram, y_val_vram, batch_size=batch_size*2, shuffle=False)

    model = TabularMLP(input_dim=input_dimension, hidden_layers=hidden_layers, dropout_rate=dropout_rate, activation_name=activation_name).to(device)
                       
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    scaler = torch.amp.GradScaler(device_type)

    best_macro_f1 = 0.0
    patience_counter = 0
    early_stopping_patience = 10
    best_model_state = None

    for epoch in range(epochs):
        # Entrenamiento
        model.train()
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            
            with torch.amp.autocast(device_type=device_type):
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        # Validación
        model.eval()
        val_loss = 0.0
        y_pred_list = []
        
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                with torch.amp.autocast(device_type=device_type):
                    outputs = model(batch_X)
                    loss = criterion(outputs, batch_y)
                    
                val_loss += loss.item() * batch_X.size(0)
                _, preds = torch.max(outputs, 1)
                y_pred_list.append(preds)
                
        val_loss = val_loss / len(X_val_vram)
        scheduler.step(val_loss)
        
        y_pred_all = torch.cat(y_pred_list).cpu().numpy()
        current_macro_f1 = f1_score(y_val_cpu, y_pred_all, average='macro', zero_division=0)
        
        trial.report(current_macro_f1, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
            
        # Early Stopping
        if current_macro_f1 > best_macro_f1:
            best_macro_f1 = current_macro_f1
            patience_counter = 0
            best_model_state = copy.deepcopy(model.state_dict())
        else:
            patience_counter += 1
            
        if patience_counter >= early_stopping_patience:
            break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        
    model.eval()
    with torch.no_grad():
        val_logits = model(X_val_vram)
        final_preds = torch.argmax(val_logits, dim=1).cpu().numpy()

    final_f1_mac = f1_score(y_val_cpu, final_preds, average='macro')
    report = classification_report(y_val_cpu, final_preds, zero_division=0)
    
    save_confusion_matrix_mlp(y_val_cpu, final_preds, trial.number)
    
    log_msg = f"""Trial {trial.number}
MLP Baseline Ampliado
Hiperparámetros: {trial.params}

F1 Macro: {final_f1_mac:.4f}
{report}"""
    
    with open(f"{log_folder}/Trial_{trial.number}_Metrics.log", "w", encoding='utf-8') as f:
        f.write(log_msg)
    
    print(f"Trial {trial.number} finalizado | Mejor F1 Macro: {final_f1_mac:.4f}")
    
    del model, optimizer, val_logits, best_model_state
    gc.collect()
    torch.cuda.empty_cache()
    
    return final_f1_mac

print("Iniciando 100 Trials para MLP Baseline...")
study_mlp = optuna.create_study(direction='maximize', study_name="MLP_Baseline")
study_mlp.optimize(objective_mlp, n_trials=100)

print("Resultados Entrenamiento MLP Baseline")
print(f"Mejor F1 Macro: {study_mlp.best_value:.4f}")
print(f"Mejores Hiperparámetros:\n{study_mlp.best_params}")

with open(f"{log_folder}/Resumen_Final_MLP_Baseline.txt", 'w', encoding='utf-8') as f:
    f.write(f"Resultados Finales MLP Baseline (Dataset: None)\n")
    f.write(f"Mejor F1 Macro: {study_mlp.best_value:.4f} (Trial {study_mlp.best_trial.number})\n")
    f.write(f"Mejores Hiperparámetros:\n{study_mlp.best_params}\n")

In [12]:
import os
import gc
import copy
import warnings
import optuna
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from torch.optim.lr_scheduler import ReduceLROnPlateau

warnings.filterwarnings("ignore")

os.environ["CUDA_VISIBLE_DEVICES"] = "1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device_type = 'cuda' if torch.cuda.is_available() else 'cpu'

log_folder = "Logs_MLP_Baseline_2_Nuevo"
os.makedirs(log_folder, exist_ok=True)

def save_confusion_matrix_mlp_exp2(y_true, y_pred, trial_number, dataset_name, phase="Val"):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Purples')
    plt.title(f'CM MLP - {dataset_name.upper()} - Trial {trial_number} ({phase})')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig(f'{log_folder}/{dataset_name}_Trial_{trial_number}_MLP_CM.png')
    plt.close()

class TensorDataLoader:
    def __init__(self, X, y, batch_size, shuffle=False):
        self.X = X
        self.y = y
        self.dataset_len = self.X.shape[0]
        self.batch_size = batch_size
        self.shuffle = shuffle

    def __iter__(self):
        if self.shuffle:
            self.indices = torch.randperm(self.dataset_len, device=device)
        else:
            self.indices = None
        self.i = 0
        return self

    def __next__(self):
        if self.i >= self.dataset_len:
            raise StopIteration
        
        if self.indices is not None:
            idx = self.indices[self.i : self.i + self.batch_size]
            batch_X = self.X[idx]
            batch_y = self.y[idx]
        else:
            batch_X = self.X[self.i : self.i + self.batch_size]
            batch_y = self.y[self.i : self.i + self.batch_size]
            
        self.i += self.batch_size
        return batch_X, batch_y

    def __len__(self):
        return (self.dataset_len + self.batch_size - 1) // self.batch_size

class TabularMLP(nn.Module):
    def __init__(self, input_dim, hidden_layers, dropout_rate, activation_name, num_classes=15):
        super().__init__()
        activations = {
            'leaky_relu': nn.LeakyReLU(0.01),
            'gelu': nn.GELU(),
            'relu': nn.ReLU()
        }
        activation_layer = activations[activation_name]

        layers = []
        in_f = input_dim
        
        for out_f in hidden_layers:
            layers.append(nn.Linear(in_f, out_f))
            layers.append(nn.BatchNorm1d(out_f))
            layers.append(activation_layer)
            layers.append(nn.Dropout(dropout_rate))
            in_f = out_f
        
        layers.append(nn.Linear(in_f, num_classes))
        self.net = nn.Sequential(*layers)
        
    def forward(self, x): 
        return self.net(x)

print("Preparando tensor de validación en VRAM...")
X_val_vram = torch.tensor(np.array(X_val, dtype=np.float32)).to(device)
y_val_vram = torch.tensor(np.array(y_val_1d, dtype=np.int64)).to(device)
y_val_cpu = y_val_vram.cpu().numpy()

datasets_to_evaluate = ['smote', 'smote_enn', 'smote_tomek']

for ds_name in datasets_to_evaluate:
    print(f"MLP - Experimento 2: Iniciando Dataset {ds_name.upper()}")
    
    x_raw, y_raw = train_datasets[ds_name]
    x_train_vram = torch.tensor(np.array(x_raw, dtype=np.float32)).to(device)
    y_train_vram = torch.tensor(np.array(y_raw, dtype=np.int64)).to(device)

    input_dimension = x_train_vram.shape[1]

    def objective_mlp_exp2(trial):
        batch_size = trial.suggest_categorical('batch_size', [4096, 8192, 16384])
        dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)
        activation_name = trial.suggest_categorical('activation', ['leaky_relu', 'gelu', 'relu'])
        lr = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
        weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True)
        epochs = trial.suggest_int('epochs', 50, 100)
        
        n_layers = trial.suggest_int('n_layers', 2, 4)
        shape_strategy = trial.suggest_categorical('mlp_shape', ['bottleneck', 'flat'])
        
        hidden_layers = []
        base_units = trial.suggest_int('n_units_l0', 256, 1024, step=256)
        hidden_layers.append(base_units)
        
        if shape_strategy == 'flat':
            for _ in range(1, n_layers):
                hidden_layers.append(base_units)
        else:
            prev_units = base_units
            for i in range(1, n_layers):
                units = trial.suggest_int(f'n_units_l{i}', 64, prev_units, step=64)
                hidden_layers.append(units)
                prev_units = units
                
        train_loader = TensorDataLoader(x_train_vram, y_train_vram, batch_size=batch_size, shuffle=True)
        val_loader = TensorDataLoader(X_val_vram, y_val_vram, batch_size=batch_size*2, shuffle=False)

        model = TabularMLP(input_dim=input_dimension, hidden_layers=hidden_layers, dropout_rate=dropout_rate, activation_name=activation_name).to(device)
                           
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
        scaler = torch.amp.GradScaler(device_type)

        best_macro_f1 = 0.0
        patience_counter = 0
        early_stopping_patience = 10
        best_model_state = None

        for epoch in range(epochs):
            # Entrenamiento
            model.train()
            for batch_X, batch_y in train_loader:
                optimizer.zero_grad()
                
                with torch.amp.autocast(device_type=device_type):
                    outputs = model(batch_X)
                    loss = criterion(outputs, batch_y)
                    
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

            # Validación
            model.eval()
            val_loss = 0.0
            y_pred_list = []
            
            with torch.no_grad():
                for batch_X, batch_y in val_loader:
                    with torch.amp.autocast(device_type=device_type):
                        outputs = model(batch_X)
                        loss = criterion(outputs, batch_y)
                        
                    val_loss += loss.item() * batch_X.size(0)
                    _, preds = torch.max(outputs, 1)
                    y_pred_list.append(preds)
                    
            val_loss = val_loss / len(X_val_vram)
            scheduler.step(val_loss)
            
            y_pred_all = torch.cat(y_pred_list).cpu().numpy()
            current_macro_f1 = f1_score(y_val_cpu, y_pred_all, average='macro', zero_division=0)
            
            trial.report(current_macro_f1, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()
                
            # Early Stopping
            if current_macro_f1 > best_macro_f1:
                best_macro_f1 = current_macro_f1
                patience_counter = 0
                best_model_state = copy.deepcopy(model.state_dict())
            else:
                patience_counter += 1
                
            if patience_counter >= early_stopping_patience:
                break

        if best_model_state is not None:
            model.load_state_dict(best_model_state)
            
        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_vram)
            final_preds = torch.argmax(val_logits, dim=1).cpu().numpy()

        final_f1_mac = f1_score(y_val_cpu, final_preds, average='macro')
        report = classification_report(y_val_cpu, final_preds, zero_division=0)
        
        save_confusion_matrix_mlp_exp2(y_val_cpu, final_preds, trial.number, ds_name)
        
        log_msg = f"""Trial {trial.number}
Experimento 2 MLP: Oversampling ({ds_name})
Hiperparámetros: {trial.params}

F1 Macro: {final_f1_mac:.4f}
{report}"""
        
        with open(f"{log_folder}/{ds_name}_Trial_{trial.number}_Metrics.log", "w", encoding='utf-8') as f:
            f.write(log_msg)
        
        print(f"[{ds_name.upper()}] Trial {trial.number} finalizado | Mejor F1 Macro: {final_f1_mac:.4f}")
        
        del model, optimizer, val_logits, best_model_state
        gc.collect()
        torch.cuda.empty_cache()
        
        return final_f1_mac

    study_name = f"MLP_Exp2_{ds_name}"
    study_mlp = optuna.create_study(direction='maximize', study_name=study_name)
    study_mlp.optimize(objective_mlp_exp2, n_trials=50)

    print(f"\nResultados para {ds_name.upper()}:")
    print(f"Mejor Trial: {study_mlp.best_trial.number}")
    print(f"Mejor F1 Macro: {study_mlp.best_value:.4f}")
    
    with open(f"{log_folder}/Resumen_Exp2_Oversampling.txt", 'a', encoding='utf-8') as res_file:
        res_file.write(f"Dataset: {ds_name.upper()}\n")
        res_file.write(f"Mejor F1 Macro: {study_mlp.best_value:.4f} (Trial {study_mlp.best_trial.number})\n")
        res_file.write(f"Params: {study_mlp.best_params}\n\n")

    del x_train_vram, y_train_vram
    gc.collect()
    torch.cuda.empty_cache()

print("\nExperimento 2 (MLP + SMOTE Variants) Completado")

Preparando tensor de validación en VRAM...
MLP - Experimento 2: Iniciando Dataset SMOTE


[I 2026-04-11 03:03:46,483] A new study created in memory with name: MLP_Exp2_smote
[I 2026-04-11 03:04:52,029] Trial 0 finished with value: 0.6858445021897369 and parameters: {'batch_size': 4096, 'dropout_rate': 0.15585078966826213, 'activation': 'relu', 'lr': 0.00010402465851666061, 'weight_decay': 4.56645110612025e-06, 'epochs': 83, 'n_layers': 4, 'mlp_shape': 'bottleneck', 'n_units_l0': 256, 'n_units_l1': 64, 'n_units_l2': 64, 'n_units_l3': 64}. Best is trial 0 with value: 0.6858445021897369.


[SMOTE] Trial 0 finalizado | Mejor F1 Macro: 0.6858


[I 2026-04-11 03:07:37,207] Trial 1 finished with value: 0.7212912744296017 and parameters: {'batch_size': 16384, 'dropout_rate': 0.4609751046319974, 'activation': 'relu', 'lr': 0.001684700687834111, 'weight_decay': 0.00044191484592245465, 'epochs': 58, 'n_layers': 3, 'mlp_shape': 'flat', 'n_units_l0': 768}. Best is trial 1 with value: 0.7212912744296017.


[SMOTE] Trial 1 finalizado | Mejor F1 Macro: 0.7213


...

[SMOTE_TOMEK] Trial 47 finalizado | Mejor F1 Macro: 0.7551


[I 2026-04-11 06:55:13,147] Trial 48 finished with value: 0.7428501303906413 and parameters: {'batch_size': 4096, 'dropout_rate': 0.20045446822942942, 'activation': 'relu', 'lr': 0.0025846423270723964, 'weight_decay': 2.849003927580969e-05, 'epochs': 76, 'n_layers': 3, 'mlp_shape': 'bottleneck', 'n_units_l0': 768, 'n_units_l1': 576, 'n_units_l2': 512}. Best is trial 33 with value: 0.7919374773736133.


[SMOTE_TOMEK] Trial 48 finalizado | Mejor F1 Macro: 0.7429


[I 2026-04-11 07:00:42,638] Trial 49 finished with value: 0.7450454999208564 and parameters: {'batch_size': 4096, 'dropout_rate': 0.22071118870997078, 'activation': 'relu', 'lr': 0.0018893810754185127, 'weight_decay': 5.626381967431192e-05, 'epochs': 83, 'n_layers': 3, 'mlp_shape': 'flat', 'n_units_l0': 768}. Best is trial 33 with value: 0.7919374773736133.


[SMOTE_TOMEK] Trial 49 finalizado | Mejor F1 Macro: 0.7450

Resultados para SMOTE_TOMEK:
Mejor Trial: 33
Mejor F1 Macro: 0.7919

Experimento 2 (MLP + SMOTE Variants) Completado


In [13]:
import os
import gc
import copy
import warnings
import optuna
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from torch.optim.lr_scheduler import ReduceLROnPlateau

warnings.filterwarnings("ignore")

os.environ["CUDA_VISIBLE_DEVICES"] = "1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device_type = 'cuda' if torch.cuda.is_available() else 'cpu'

log_folder = "Logs_MLP_Baseline_3_Nuevo"
os.makedirs(log_folder, exist_ok=True)

def save_confusion_matrix_mlp_exp3(y_true, y_pred, trial_number, weight_name, phase="Val"):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Purples')
    plt.title(f'CM MLP - Weight: {weight_name.upper()} - Trial {trial_number} ({phase})')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig(f'{log_folder}/Weight_{weight_name}_Trial_{trial_number}_MLP_CM.png')
    plt.close()

class TensorDataLoader:
    def __init__(self, X, y, batch_size, shuffle=False):
        self.X = X
        self.y = y
        self.dataset_len = self.X.shape[0]
        self.batch_size = batch_size
        self.shuffle = shuffle

    def __iter__(self):
        if self.shuffle:
            self.indices = torch.randperm(self.dataset_len, device=device)
        else:
            self.indices = None
        self.i = 0
        return self

    def __next__(self):
        if self.i >= self.dataset_len:
            raise StopIteration
        
        if self.indices is not None:
            idx = self.indices[self.i : self.i + self.batch_size]
            batch_X = self.X[idx]
            batch_y = self.y[idx]
        else:
            batch_X = self.X[self.i : self.i + self.batch_size]
            batch_y = self.y[self.i : self.i + self.batch_size]
            
        self.i += self.batch_size
        return batch_X, batch_y

    def __len__(self):
        return (self.dataset_len + self.batch_size - 1) // self.batch_size

class TabularMLP(nn.Module):
    def __init__(self, input_dim, hidden_layers, dropout_rate, activation_name, num_classes=15):
        super().__init__()
        activations = {
            'leaky_relu': nn.LeakyReLU(0.01),
            'gelu': nn.GELU(),
            'relu': nn.ReLU()
        }
        activation_layer = activations[activation_name]

        layers = []
        in_f = input_dim
        
        for out_f in hidden_layers:
            layers.append(nn.Linear(in_f, out_f))
            layers.append(nn.BatchNorm1d(out_f))
            layers.append(activation_layer)
            layers.append(nn.Dropout(dropout_rate))
            in_f = out_f
        
        layers.append(nn.Linear(in_f, num_classes))
        self.net = nn.Sequential(*layers)
        
    def forward(self, x): 
        return self.net(x)

print("Preparando tensores en VRAM...")
X_val_vram = torch.tensor(np.array(X_val, dtype=np.float32)).to(device)
y_val_vram = torch.tensor(np.array(y_val_1d, dtype=np.int64)).to(device)
y_val_cpu = y_val_vram.cpu().numpy()

x_raw, y_raw = train_datasets['none'] 
x_train_vram = torch.tensor(np.array(x_raw, dtype=np.float32)).to(device)
y_train_vram = torch.tensor(np.array(y_raw, dtype=np.int64)).to(device)

input_dimension = x_train_vram.shape[1]
weight_methods = ['balanced', 'polynomial', 'effective_samples']

for w_name in weight_methods:
    print(f"MLP - Experimento 3: Iniciando Pesos {w_name.upper()}")

    def objective_mlp_exp3(trial):
        batch_size = trial.suggest_categorical('batch_size', [4096, 8192, 16384])
        dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)
        activation_name = trial.suggest_categorical('activation', ['leaky_relu', 'gelu', 'relu'])
        lr = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
        weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True)
        epochs = trial.suggest_int('epochs', 50, 100)
        
        n_layers = trial.suggest_int('n_layers', 2, 4)
        shape_strategy = trial.suggest_categorical('mlp_shape', ['bottleneck', 'flat'])
        
        if w_name == 'balanced':
            w_dict = balanced_class_weight(y_raw)
        elif w_name == 'polynomial':
            alpha_val = trial.suggest_float('poly_alpha', 0.1, 1.0)
            w_dict = polynomial_class_weight(y_raw, alpha=alpha_val)
        elif w_name == 'effective_samples':
            beta_val = trial.suggest_float('beta_eff_samp', 0.9, 0.9999)
            w_dict = effective_samples_class_weight(y_raw, beta=beta_val)
        elif w_name == 'focal':
            gamma_val = trial.suggest_float('focal_gamma', 0.5, 5.0)
            w_dict = focal_class_weight_improved(y_raw, gamma=gamma_val)
            
        peso_lista = [w_dict.get(i, 1.0) for i in range(15)]
        current_weight_tensor = torch.tensor(peso_lista, dtype=torch.float32).to(device)

        hidden_layers = []
        base_units = trial.suggest_int('n_units_l0', 256, 1024, step=256)
        hidden_layers.append(base_units)
        
        if shape_strategy == 'flat':
            for _ in range(1, n_layers):
                hidden_layers.append(base_units)
        else:
            prev_units = base_units
            for i in range(1, n_layers):
                units = trial.suggest_int(f'n_units_l{i}', 64, prev_units, step=64)
                hidden_layers.append(units)
                prev_units = units

        # Dataloaders
        train_loader = TensorDataLoader(x_train_vram, y_train_vram, batch_size=batch_size, shuffle=True)
        val_loader = TensorDataLoader(X_val_vram, y_val_vram, batch_size=batch_size*2, shuffle=False)

        model = TabularMLP(input_dim=input_dimension, hidden_layers=hidden_layers, 
                           dropout_rate=dropout_rate, activation_name=activation_name,
                           num_classes=15).to(device)
                           
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        
        criterion = nn.CrossEntropyLoss(weight=current_weight_tensor)
        scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
        scaler = torch.amp.GradScaler(device_type)

        best_macro_f1 = 0.0
        patience_counter = 0
        early_stopping_patience = 10
        best_model_state = None

        for epoch in range(epochs):
            # Entrenamiento
            model.train()
            for batch_X, batch_y in train_loader:
                optimizer.zero_grad()
                
                with torch.amp.autocast(device_type=device_type):
                    outputs = model(batch_X)
                    loss = criterion(outputs, batch_y)
                    
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

            # Validación
            model.eval()
            val_loss = 0.0
            y_pred_list = []
            
            with torch.no_grad():
                for batch_X, batch_y in val_loader:
                    with torch.amp.autocast(device_type=device_type):
                        outputs = model(batch_X)
                        loss = criterion(outputs, batch_y)
                        
                    val_loss += loss.item() * batch_X.size(0)
                    _, preds = torch.max(outputs, 1)
                    y_pred_list.append(preds)
                    
            val_loss = val_loss / len(X_val_vram)
            scheduler.step(val_loss)
            
            y_pred_all = torch.cat(y_pred_list).cpu().numpy()
            current_macro_f1 = f1_score(y_val_cpu, y_pred_all, average='macro', zero_division=0)
            
            trial.report(current_macro_f1, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()
                
            # Early Stopping
            if current_macro_f1 > best_macro_f1:
                best_macro_f1 = current_macro_f1
                patience_counter = 0
                best_model_state = copy.deepcopy(model.state_dict())
            else:
                patience_counter += 1
                
            if patience_counter >= early_stopping_patience:
                break

        if best_model_state is not None:
            model.load_state_dict(best_model_state)
            
        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_vram)
            final_preds = torch.argmax(val_logits, dim=1).cpu().numpy()

        final_f1_mac = f1_score(y_val_cpu, final_preds, average='macro')
        report = classification_report(y_val_cpu, final_preds, zero_division=0)
        
        save_confusion_matrix_mlp_exp3(y_val_cpu, final_preds, trial.number, w_name)
        
        log_msg = f"""Trial {trial.number}
Exp 3 MLP: Weights ({w_name})
Params: {trial.params}

F1 Macro: {final_f1_mac:.4f}
{report}"""
        
        with open(f"{log_folder}/{w_name}_Trial_{trial.number}.log", "w", encoding='utf-8') as f:
            f.write(log_msg)
        
        print(f"[{w_name.upper()}] Trial {trial.number} finalizado | Mejor F1: {final_f1_mac:.4f}")
        
        del model, optimizer, val_logits, best_model_state, criterion, current_weight_tensor
        gc.collect()
        torch.cuda.empty_cache()
        
        return final_f1_mac

    study_name = f"MLP_Exp3_{w_name}"
    study_mlp = optuna.create_study(direction='maximize', study_name=study_name)
    study_mlp.optimize(objective_mlp_exp3, n_trials=50)

    print(f"\nResultados para {w_name.upper()}:")
    print(f"Mejor Trial: {study_mlp.best_trial.number}")
    print(f"Mejor F1 Macro: {study_mlp.best_value:.4f}")
    
    with open(f"{log_folder}/Resumen_Exp3_Weights.txt", 'a', encoding='utf-8') as res_file:
        res_file.write(f"Método de Peso: {w_name.upper()}\n")
        res_file.write(f"Mejor F1 Macro: {study_mlp.best_value:.4f} (Trial {study_mlp.best_trial.number})\n")
        res_file.write(f"Params: {study_mlp.best_params}\n\n")

print("\nExperimento 3 (Funciones de Pesos) Completado")

Preparando tensores en VRAM...


[I 2026-04-11 07:00:43,548] A new study created in memory with name: MLP_Exp3_balanced


MLP - Experimento 3: Iniciando Pesos BALANCED
[BALANCED] Trial 0 finalizado | Mejor F1: 0.4322


[I 2026-04-11 07:01:45,588] Trial 0 finished with value: 0.4321564060343816 and parameters: {'batch_size': 8192, 'dropout_rate': 0.10655516934213392, 'activation': 'relu', 'lr': 0.0041964072672422215, 'weight_decay': 4.476976668301873e-05, 'epochs': 56, 'n_layers': 2, 'mlp_shape': 'bottleneck', 'n_units_l0': 768, 'n_units_l1': 320}. Best is trial 0 with value: 0.4321564060343816.
[I 2026-04-11 07:05:45,375] Trial 1 finished with value: 0.43086234781105825 and parameters: {'batch_size': 16384, 'dropout_rate': 0.4261172803853054, 'activation': 'gelu', 'lr': 0.001109737425570131, 'weight_decay': 0.00021657943852855402, 'epochs': 90, 'n_layers': 3, 'mlp_shape': 'flat', 'n_units_l0': 768}. Best is trial 0 with value: 0.4321564060343816.


[BALANCED] Trial 1 finalizado | Mejor F1: 0.4309


...

[EFFECTIVE_SAMPLES] Trial 29 finalizado | Mejor F1: 0.7590


[I 2026-04-11 10:03:19,722] Trial 30 pruned. 
[I 2026-04-11 10:04:17,877] Trial 31 finished with value: 0.7480860246474036 and parameters: {'batch_size': 8192, 'dropout_rate': 0.31832829857244105, 'activation': 'leaky_relu', 'lr': 0.0041179058386320855, 'weight_decay': 9.366473829104916e-06, 'epochs': 53, 'n_layers': 2, 'mlp_shape': 'flat', 'beta_eff_samp': 0.9364399937310369, 'n_units_l0': 512}. Best is trial 2 with value: 0.761963219748153.


[EFFECTIVE_SAMPLES] Trial 31 finalizado | Mejor F1: 0.7481


[I 2026-04-11 10:04:28,360] Trial 32 pruned. 
[I 2026-04-11 10:04:35,378] Trial 33 pruned. 
[I 2026-04-11 10:04:40,569] Trial 34 pruned. 
[I 2026-04-11 10:04:45,627] Trial 35 pruned. 
[I 2026-04-11 10:04:56,052] Trial 36 pruned. 
[I 2026-04-11 10:04:58,735] Trial 37 pruned. 
[I 2026-04-11 10:05:02,175] Trial 38 pruned. 
[I 2026-04-11 10:05:06,405] Trial 39 pruned. 
[I 2026-04-11 10:05:14,622] Trial 40 pruned. 
[I 2026-04-11 10:05:17,300] Trial 41 pruned. 
[I 2026-04-11 10:05:19,560] Trial 42 pruned. 
[I 2026-04-11 10:05:21,169] Trial 43 pruned. 
[I 2026-04-11 10:05:37,334] Trial 44 pruned. 
[I 2026-04-11 10:05:39,929] Trial 45 pruned. 
[I 2026-04-11 10:05:50,171] Trial 46 pruned. 
[I 2026-04-11 10:06:05,553] Trial 47 pruned. 
[I 2026-04-11 10:06:07,917] Trial 48 pruned. 
[I 2026-04-11 10:06:18,119] Trial 49 pruned. 



Resultados para EFFECTIVE_SAMPLES:
Mejor Trial: 2
Mejor F1 Macro: 0.7620

Experimento 3 (Funciones de Pesos) Completado


In [14]:
import os
import gc
import copy
import warnings
import optuna
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from torch.optim.lr_scheduler import ReduceLROnPlateau

warnings.filterwarnings("ignore")

os.environ["CUDA_VISIBLE_DEVICES"] = "1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device_type = 'cuda' if torch.cuda.is_available() else 'cpu'

log_folder = "Logs_MLP_Baseline_4_Nuevo"
os.makedirs(log_folder, exist_ok=True)

def save_confusion_matrix_mlp_exp4(y_true, y_pred, trial_number, method_name, phase="Val"):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Purples')
    plt.title(f'CM MLP - FS: {method_name.upper()} - Trial {trial_number} ({phase})')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig(f'{log_folder}/{method_name}_Trial_{trial_number}_MLP_CM.png')
    plt.close()

class TensorDataLoader:
    def __init__(self, X, y, batch_size, shuffle=False):
        self.X = X
        self.y = y
        self.dataset_len = self.X.shape[0]
        self.batch_size = batch_size
        self.shuffle = shuffle

    def __iter__(self):
        if self.shuffle:
            self.indices = torch.randperm(self.dataset_len, device=device)
        else:
            self.indices = None
        self.i = 0
        return self

    def __next__(self):
        if self.i >= self.dataset_len:
            raise StopIteration
        
        if self.indices is not None:
            idx = self.indices[self.i : self.i + self.batch_size]
            batch_X = self.X[idx]
            batch_y = self.y[idx]
        else:
            batch_X = self.X[self.i : self.i + self.batch_size]
            batch_y = self.y[self.i : self.i + self.batch_size]
            
        self.i += self.batch_size
        return batch_X, batch_y

    def __len__(self):
        return (self.dataset_len + self.batch_size - 1) // self.batch_size

class TabularMLP(nn.Module):
    def __init__(self, input_dim, hidden_layers, dropout_rate, activation_name, num_classes=15):
        super().__init__()
        activations = {
            'leaky_relu': nn.LeakyReLU(0.01),
            'gelu': nn.GELU(),
            'relu': nn.ReLU()
        }
        activation_layer = activations[activation_name]

        layers = []
        in_f = input_dim
        
        for out_f in hidden_layers:
            layers.append(nn.Linear(in_f, out_f))
            layers.append(nn.BatchNorm1d(out_f))
            layers.append(activation_layer)
            layers.append(nn.Dropout(dropout_rate))
            in_f = out_f
        
        layers.append(nn.Linear(in_f, num_classes))
        self.net = nn.Sequential(*layers)
        
    def forward(self, x): 
        return self.net(x)

print("Preparando datos base...")
X_val_raw_cpu = np.array(X_val, dtype=np.float32)
y_val_vram = torch.tensor(np.array(y_val_1d, dtype=np.int64)).to(device)
y_val_cpu = y_val_vram.cpu().numpy()

x_raw_train_cpu, y_raw_train_cpu = train_datasets['none']
total_columns = x_raw_train_cpu.shape[1]

y_train_vram = torch.tensor(np.array(y_raw_train_cpu, dtype=np.int64)).to(device)

fs_methods = ['f_classif', 'tree']

for method in fs_methods:
    print(f"MLP - Experimento 4: Iniciando Selección {method.upper()}")

    def objective_mlp_exp4(trial):
        k_features = trial.suggest_int('k_features', 30, total_columns)
        
        selector = FeatureSelector(method=method, k=k_features)
        
        X_train_fs = selector.fit_transform(x_raw_train_cpu, y_raw_train_cpu)
        X_val_fs = selector.transform(X_val_raw_cpu)
        
        X_train_fs_vram = torch.tensor(np.array(X_train_fs), dtype=torch.float32).to(device)
        X_val_fs_vram = torch.tensor(np.array(X_val_fs), dtype=torch.float32).to(device)
        
        batch_size = trial.suggest_categorical('batch_size', [4096, 8192, 16384])
        dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)
        activation_name = trial.suggest_categorical('activation', ['leaky_relu', 'gelu', 'relu'])
        lr = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
        weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True)
        epochs = trial.suggest_int('epochs', 50, 100)
        
        n_layers = trial.suggest_int('n_layers', 2, 4)
        shape_strategy = trial.suggest_categorical('mlp_shape', ['bottleneck', 'flat'])
        
        hidden_layers = []
        base_units = trial.suggest_int('n_units_l0', 256, 1024, step=256)
        hidden_layers.append(base_units)
        
        if shape_strategy == 'flat':
            for _ in range(1, n_layers):
                hidden_layers.append(base_units)
        else:
            prev_units = base_units
            for i in range(1, n_layers):
                units = trial.suggest_int(f'n_units_l{i}', 64, prev_units, step=64)
                hidden_layers.append(units)
                prev_units = units

        train_loader = TensorDataLoader(X_train_fs_vram, y_train_vram, batch_size=batch_size, shuffle=True)
        val_loader = TensorDataLoader(X_val_fs_vram, y_val_vram, batch_size=batch_size*2, shuffle=False)

        model = TabularMLP(input_dim=k_features, hidden_layers=hidden_layers, 
                           dropout_rate=dropout_rate, activation_name=activation_name,
                           num_classes=15).to(device)
                           
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
        scaler = torch.amp.GradScaler(device_type)

        best_macro_f1 = 0.0
        patience_counter = 0
        early_stopping_patience = 10
        best_model_state = None

        for epoch in range(epochs):
            model.train()
            for batch_X, batch_y in train_loader:
                optimizer.zero_grad()
                
                with torch.amp.autocast(device_type=device_type):
                    outputs = model(batch_X)
                    loss = criterion(outputs, batch_y)
                    
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

            # Validación
            model.eval()
            val_loss = 0.0
            y_pred_list = []
            
            with torch.no_grad():
                for batch_X, batch_y in val_loader:
                    with torch.amp.autocast(device_type=device_type):
                        outputs = model(batch_X)
                        loss = criterion(outputs, batch_y)
                        
                    val_loss += loss.item() * batch_X.size(0)
                    _, preds = torch.max(outputs, 1)
                    y_pred_list.append(preds)
                    
            val_loss = val_loss / len(X_val_fs_vram)
            scheduler.step(val_loss)
            
            y_pred_all = torch.cat(y_pred_list).cpu().numpy()
            current_macro_f1 = f1_score(y_val_cpu, y_pred_all, average='macro', zero_division=0)
            
            # Pruning
            trial.report(current_macro_f1, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()
                
            # Early Stopping
            if current_macro_f1 > best_macro_f1:
                best_macro_f1 = current_macro_f1
                patience_counter = 0
                best_model_state = copy.deepcopy(model.state_dict())
            else:
                patience_counter += 1
                
            if patience_counter >= early_stopping_patience:
                break

        if best_model_state is not None:
            model.load_state_dict(best_model_state)
            
        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_fs_vram)
            final_preds = torch.argmax(val_logits, dim=1).cpu().numpy()

        final_f1_mac = f1_score(y_val_cpu, final_preds, average='macro')
        report = classification_report(y_val_cpu, final_preds, zero_division=0)
        
        save_confusion_matrix_mlp_exp4(y_val_cpu, final_preds, trial.number, method)
        
        log_msg = f"""Trial {trial.number}
Exp 4 MLP: Feature Selection ({method})
Params FS (k): {k_features}
Params MLP: {trial.params}

F1 Macro: {final_f1_mac:.4f}
{report}"""
        
        with open(f"{log_folder}/{method}_Trial_{trial.number}.log", "w", encoding='utf-8') as f:
            f.write(log_msg)
        
        print(f"[{method.upper()} | k={k_features}] Trial {trial.number} finalizado | Mejor F1: {final_f1_mac:.4f}")
        
        del model, optimizer, val_logits, best_model_state
        del X_train_fs_vram, X_val_fs_vram, X_train_fs, X_val_fs
        gc.collect()
        torch.cuda.empty_cache()
        
        return final_f1_mac

    study_name = f"MLP_Exp4_{method}"
    study_mlp = optuna.create_study(direction='maximize', study_name=study_name)
    study_mlp.optimize(objective_mlp_exp4, n_trials=50)

    print(f"\nResultados para {method.upper()}:")
    print(f"Mejor F1 Macro: {study_mlp.best_value:.4f} (Trial {study_mlp.best_trial.number})")
    
    with open(f"{log_folder}/Resumen_Exp4_FS.txt", 'a', encoding='utf-8') as res_file:
        res_file.write(f"FS Método: {method.upper()}\n")
        res_file.write(f"Mejor F1 Macro: {study_mlp.best_value:.4f} (Trial {study_mlp.best_trial.number})\n")
        res_file.write(f"Params: {study_mlp.best_params}\n\n")

print("\nExperimento 4 (Selección de Características) Completado")

[I 2026-04-11 10:06:18,253] A new study created in memory with name: MLP_Exp4_f_classif


Preparando datos base...
MLP - Experimento 4: Iniciando Selección F_CLASSIF


[I 2026-04-11 10:12:06,761] Trial 0 finished with value: 0.5902794007695278 and parameters: {'k_features': 37, 'batch_size': 8192, 'dropout_rate': 0.4465377846423618, 'activation': 'relu', 'lr': 0.002212625685856239, 'weight_decay': 2.509540783391964e-05, 'epochs': 94, 'n_layers': 4, 'mlp_shape': 'bottleneck', 'n_units_l0': 1024, 'n_units_l1': 128, 'n_units_l2': 64, 'n_units_l3': 64}. Best is trial 0 with value: 0.5902794007695278.


[F_CLASSIF | k=37] Trial 0 finalizado | Mejor F1: 0.5903
[F_CLASSIF | k=57] Trial 1 finalizado | Mejor F1: 0.6307


[I 2026-04-11 10:15:06,626] Trial 1 finished with value: 0.630727312640675 and parameters: {'k_features': 57, 'batch_size': 4096, 'dropout_rate': 0.3714652698973461, 'activation': 'leaky_relu', 'lr': 0.0011967979753605192, 'weight_decay': 0.00018417563136004177, 'epochs': 97, 'n_layers': 4, 'mlp_shape': 'bottleneck', 'n_units_l0': 768, 'n_units_l1': 448, 'n_units_l2': 64, 'n_units_l3': 64}. Best is trial 1 with value: 0.630727312640675.


...

[TREE | k=62] Trial 43 finalizado | Mejor F1: 0.7572


[I 2026-04-11 13:26:24,717] Trial 44 pruned. 
[I 2026-04-11 13:26:39,276] Trial 45 pruned. 
[I 2026-04-11 13:26:55,996] Trial 46 pruned. 
[I 2026-04-11 13:27:09,282] Trial 47 pruned. 


[TREE | k=63] Trial 48 finalizado | Mejor F1: 0.7629


[I 2026-04-11 13:32:02,520] Trial 48 finished with value: 0.7628547844099237 and parameters: {'k_features': 63, 'batch_size': 4096, 'dropout_rate': 0.2967883754445601, 'activation': 'gelu', 'lr': 0.0014028733822867442, 'weight_decay': 0.0001251634666544587, 'epochs': 52, 'n_layers': 2, 'mlp_shape': 'flat', 'n_units_l0': 768}. Best is trial 7 with value: 0.7676938388835826.
[I 2026-04-11 13:34:17,832] Trial 49 finished with value: 0.7544667196437621 and parameters: {'k_features': 62, 'batch_size': 4096, 'dropout_rate': 0.28660214027779735, 'activation': 'leaky_relu', 'lr': 0.001441086235334084, 'weight_decay': 4.449391449515607e-05, 'epochs': 62, 'n_layers': 2, 'mlp_shape': 'flat', 'n_units_l0': 768}. Best is trial 7 with value: 0.7676938388835826.


[TREE | k=62] Trial 49 finalizado | Mejor F1: 0.7545

Resultados para TREE:
Mejor F1 Macro: 0.7677 (Trial 7)

Experimento 4 (Selección de Características) Completado


In [15]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.gamma = gamma

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, reduction='none')
        # Probabilidades de las clases correctas
        pt = torch.exp(-ce_loss)
        # Factor de penalización focal
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

In [16]:
import os
import gc
import copy
import warnings
import optuna
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from torch.optim.lr_scheduler import ReduceLROnPlateau

warnings.filterwarnings("ignore")

os.environ["CUDA_VISIBLE_DEVICES"] = "1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device_type = 'cuda' if torch.cuda.is_available() else 'cpu'

log_folder = "Logs_MLP_Baseline_5_Nuevo"
os.makedirs(log_folder, exist_ok=True)

def save_confusion_matrix_mlp_exp5(y_true, y_pred, trial_number, loss_name, phase="Val"):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Purples')
    plt.title(f'CM MLP - Loss: {loss_name.upper()} - Trial {trial_number} ({phase})')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig(f'{log_folder}/{loss_name}_Trial_{trial_number}_MLP_CM.png')
    plt.close()

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.gamma = gamma

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, reduction='none')
        # Probabilidades de las clases correctas
        pt = torch.exp(-ce_loss)
        # Factor de penalización focal
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

class TensorDataLoader:
    def __init__(self, X, y, batch_size, shuffle=False):
        self.X = X
        self.y = y
        self.dataset_len = self.X.shape[0]
        self.batch_size = batch_size
        self.shuffle = shuffle

    def __iter__(self):
        if self.shuffle:
            self.indices = torch.randperm(self.dataset_len, device=device)
        else:
            self.indices = None
        self.i = 0
        return self

    def __next__(self):
        if self.i >= self.dataset_len:
            raise StopIteration
        
        if self.indices is not None:
            idx = self.indices[self.i : self.i + self.batch_size]
            batch_X = self.X[idx]
            batch_y = self.y[idx]
        else:
            batch_X = self.X[self.i : self.i + self.batch_size]
            batch_y = self.y[self.i : self.i + self.batch_size]
            
        self.i += self.batch_size
        return batch_X, batch_y

    def __len__(self):
        return (self.dataset_len + self.batch_size - 1) // self.batch_size

class TabularMLP(nn.Module):
    def __init__(self, input_dim, hidden_layers, dropout_rate, activation_name, num_classes=15):
        super().__init__()
        activations = {
            'leaky_relu': nn.LeakyReLU(0.01),
            'gelu': nn.GELU(),
            'relu': nn.ReLU()
        }
        activation_layer = activations[activation_name]

        layers = []
        in_f = input_dim
        
        for out_f in hidden_layers:
            layers.append(nn.Linear(in_f, out_f))
            layers.append(nn.BatchNorm1d(out_f))
            layers.append(activation_layer)
            layers.append(nn.Dropout(dropout_rate))
            in_f = out_f
        
        layers.append(nn.Linear(in_f, num_classes))
        self.net = nn.Sequential(*layers)
        
    def forward(self, x): 
        return self.net(x)

print("Preparando tensores en VRAM...")
X_val_vram = torch.tensor(np.array(X_val, dtype=np.float32)).to(device)
y_val_vram = torch.tensor(np.array(y_val_1d, dtype=np.int64)).to(device)
y_val_cpu = y_val_vram.cpu().numpy()

x_raw, y_raw = train_datasets['none']
x_train_vram = torch.tensor(np.array(x_raw, dtype=np.float32)).to(device)
y_train_vram = torch.tensor(np.array(y_raw, dtype=np.int64)).to(device)

input_dimension = x_train_vram.shape[1]

loss_functions = ['cross_entropy', 'focal_loss']

for loss_name in loss_functions:
    print(f"MLP - Experimento 5: Iniciando Función de Pérdida {loss_name.upper()}")

    def objective_mlp_exp5(trial):
        batch_size = trial.suggest_categorical('batch_size', [4096, 8192, 16384])
        dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)
        activation_name = trial.suggest_categorical('activation', ['leaky_relu', 'gelu', 'relu'])
        lr = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
        weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True)
        epochs = trial.suggest_int('epochs', 50, 100)
        
        n_layers = trial.suggest_int('n_layers', 2, 4)
        shape_strategy = trial.suggest_categorical('mlp_shape', ['bottleneck', 'flat'])
        
        hidden_layers = []
        base_units = trial.suggest_int('n_units_l0', 256, 1024, step=256)
        hidden_layers.append(base_units)
        
        if shape_strategy == 'flat':
            for _ in range(1, n_layers):
                hidden_layers.append(base_units)
        else:
            prev_units = base_units
            for i in range(1, n_layers):
                units = trial.suggest_int(f'n_units_l{i}', 64, prev_units, step=64)
                hidden_layers.append(units)
                prev_units = units

        # Dataloaders
        train_loader = TensorDataLoader(x_train_vram, y_train_vram, batch_size=batch_size, shuffle=True)
        val_loader = TensorDataLoader(X_val_vram, y_val_vram, batch_size=batch_size*2, shuffle=False)

        model = TabularMLP(input_dim=input_dimension, hidden_layers=hidden_layers, 
                           dropout_rate=dropout_rate, activation_name=activation_name,
                           num_classes=15).to(device)
                           
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        
        if loss_name == 'cross_entropy':
            criterion = nn.CrossEntropyLoss()
        elif loss_name == 'focal_loss':
            gamma_val = trial.suggest_float('gamma', 0.5, 5.0)
            criterion = FocalLoss(gamma=gamma_val)
            
        scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
        scaler = torch.amp.GradScaler(device_type)

        best_macro_f1 = 0.0
        patience_counter = 0
        early_stopping_patience = 10
        best_model_state = None

        for epoch in range(epochs):
            model.train()
            for batch_X, batch_y in train_loader:
                optimizer.zero_grad()
                
                with torch.amp.autocast(device_type=device_type):
                    outputs = model(batch_X)
                    loss = criterion(outputs, batch_y)
                    
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

            # Validación
            model.eval()
            val_loss = 0.0
            y_pred_list = []
            
            with torch.no_grad():
                for batch_X, batch_y in val_loader:
                    with torch.amp.autocast(device_type=device_type):
                        outputs = model(batch_X)
                        loss = criterion(outputs, batch_y)
                        
                    val_loss += loss.item() * batch_X.size(0)
                    _, preds = torch.max(outputs, 1)
                    y_pred_list.append(preds)
                    
            val_loss = val_loss / len(X_val_vram)
            scheduler.step(val_loss)
            
            y_pred_all = torch.cat(y_pred_list).cpu().numpy()
            current_macro_f1 = f1_score(y_val_cpu, y_pred_all, average='macro', zero_division=0)
            
            # Pruning
            trial.report(current_macro_f1, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()
                
            # Early Stopping
            if current_macro_f1 > best_macro_f1:
                best_macro_f1 = current_macro_f1
                patience_counter = 0
                best_model_state = copy.deepcopy(model.state_dict())
            else:
                patience_counter += 1
                
            if patience_counter >= early_stopping_patience:
                break

        if best_model_state is not None:
            model.load_state_dict(best_model_state)
            
        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_vram)
            final_preds = torch.argmax(val_logits, dim=1).cpu().numpy()

        final_f1_mac = f1_score(y_val_cpu, final_preds, average='macro')
        report = classification_report(y_val_cpu, final_preds, zero_division=0)
        
        save_confusion_matrix_mlp_exp5(y_val_cpu, final_preds, trial.number, loss_name)
        
        log_msg = f"""Trial {trial.number}
Exp 5 MLP: Loss Function ({loss_name})
Params: {trial.params}

F1 Macro: {final_f1_mac:.4f}
{report}"""
        
        with open(f"{log_folder}/{loss_name}_Trial_{trial.number}.log", "w", encoding='utf-8') as f:
            f.write(log_msg)
        
        print(f"[{loss_name.upper()}] Trial {trial.number} finalizado | Mejor F1: {final_f1_mac:.4f}")
        
        del model, optimizer, val_logits, best_model_state, criterion
        gc.collect()
        torch.cuda.empty_cache()
        
        return final_f1_mac

    study_name = f"MLP_Exp5_{loss_name}"
    study_mlp = optuna.create_study(direction='maximize', study_name=study_name)
    study_mlp.optimize(objective_mlp_exp5, n_trials=100)

    print(f"\nResultados para {loss_name.upper()}:")
    print(f"Mejor Trial: {study_mlp.best_trial.number}")
    print(f"Mejor F1 Macro: {study_mlp.best_value:.4f}")
    
    with open(f"{log_folder}/Resumen_Exp5_Loss.txt", 'a', encoding='utf-8') as res_file:
        res_file.write(f"Función de Pérdida: {loss_name.upper()}\n")
        res_file.write(f"Mejor F1 Macro: {study_mlp.best_value:.4f} (Trial {study_mlp.best_trial.number})\n")
        res_file.write(f"Params: {study_mlp.best_params}\n\n")

print("\nExperimento 5 (Funciones de Pérdida) Completado")

Preparando tensores en VRAM...


[I 2026-04-11 13:34:18,193] A new study created in memory with name: MLP_Exp5_cross_entropy


MLP - Experimento 5: Iniciando Función de Pérdida CROSS_ENTROPY


[I 2026-04-11 13:37:28,191] Trial 0 finished with value: 0.7529262800099261 and parameters: {'batch_size': 4096, 'dropout_rate': 0.41823269196928936, 'activation': 'gelu', 'lr': 0.0003873114161852696, 'weight_decay': 0.00017354246541651137, 'epochs': 79, 'n_layers': 2, 'mlp_shape': 'bottleneck', 'n_units_l0': 768, 'n_units_l1': 256}. Best is trial 0 with value: 0.7529262800099261.


[CROSS_ENTROPY] Trial 0 finalizado | Mejor F1: 0.7529


...

[FOCAL_LOSS] Trial 94 finalizado | Mejor F1: 0.7926


[I 2026-04-11 16:39:00,603] Trial 95 finished with value: 0.7879387641734019 and parameters: {'batch_size': 4096, 'dropout_rate': 0.3957197019683192, 'activation': 'leaky_relu', 'lr': 0.001570663635486656, 'weight_decay': 7.237349122357211e-06, 'epochs': 98, 'n_layers': 2, 'mlp_shape': 'flat', 'n_units_l0': 768, 'gamma': 2.2513650437105888}. Best is trial 94 with value: 0.79259593665633.


[FOCAL_LOSS] Trial 95 finalizado | Mejor F1: 0.7879


[I 2026-04-11 16:39:07,027] Trial 96 pruned. 
[I 2026-04-11 16:39:13,211] Trial 97 pruned. 
[I 2026-04-11 16:39:26,195] Trial 98 pruned. 
[I 2026-04-11 16:39:38,719] Trial 99 pruned. 



Resultados para FOCAL_LOSS:
Mejor Trial: 94
Mejor F1 Macro: 0.7926

Experimento 5 (Funciones de Pérdida) Completado


In [20]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import optuna
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import warnings
from sklearn.metrics import f1_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

log_folder = "Logs_MLP_Baseline_6"
os.makedirs(log_folder, exist_ok=True)

def save_confusion_matrix_final(y_true, y_pred, trial_number, ds_name, fs_name, w_name, l_name, phase="Val"):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'CM Final MLP - Trial {trial_number}\nDS: {ds_name} | FS: {fs_name} | W: {w_name} | Loss: {l_name}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig(f'{log_folder}/Trial_{trial_number}_Final_CM.png')
    plt.close()

class TabularMLP(nn.Module):
    def __init__(self, input_dim, n_layers, n_units, dropout_rate, activation_name, num_classes=15):
        super().__init__()
        activations = {'leaky_relu': nn.LeakyReLU(0.01), 'gelu': nn.GELU()}
        activation_layer = activations[activation_name]

        layers = []
        in_f = input_dim
        for i in range(n_layers):
            layers.append(nn.Linear(in_f, n_units))
            layers.append(nn.BatchNorm1d(n_units))
            layers.append(activation_layer)
            layers.append(nn.Dropout(dropout_rate))
            in_f = n_units
        
        layers.append(nn.Linear(in_f, num_classes))
        self.net = nn.Sequential(*layers)
        
    def forward(self, x): 
        return self.net(x)

print("Preparando tensor de validación...")
X_val_raw_cpu = np.array(X_val, dtype=np.float32)
y_val_vram = torch.tensor(np.array(y_val_1d, dtype=np.int64)).to(device)
y_val_cpu = y_val_vram.cpu().numpy()

def objective_final_mlp(trial):
    ds_name = trial.suggest_categorical('dataset', ['none', 'smote_enn'])
    x_raw_train_cpu, y_raw_train_cpu = train_datasets[ds_name]
    total_cols = x_raw_train_cpu.shape[1]
    num_samples = x_raw_train_cpu.shape[0]
    
    y_train_vram = torch.tensor(np.array(y_raw_train_cpu, dtype=np.int64)).to(device)

    fs_method = trial.suggest_categorical('fs_method', ['none', 'tree'])
    if fs_method == 'tree':
        k_opt = trial.suggest_int('k_features', 30, total_cols)
        selector = FeatureSelector(method='tree', k=k_opt)
        X_train_fs = selector.fit_transform(x_raw_train_cpu, y_raw_train_cpu)
        X_val_fs = selector.transform(X_val_raw_cpu)
    else:
        k_opt = total_cols
        X_train_fs = x_raw_train_cpu
        X_val_fs = X_val_raw_cpu

    X_train_vram = torch.tensor(np.array(X_train_fs), dtype=torch.float32).to(device)
    X_val_vram_final = torch.tensor(np.array(X_val_fs), dtype=torch.float32).to(device)

    loss_method = trial.suggest_categorical('loss_function', ['cross_entropy', 'focal_loss'])
    
    if loss_method == 'cross_entropy':
        w_method = trial.suggest_categorical('weight_method', ['none', 'polynomial'])
        
        if w_method == 'polynomial':
            alpha_val = trial.suggest_float('poly_alpha', 0.1, 1.0)
            w_dict = polynomial_class_weight(y_raw_train_cpu, alpha=alpha_val)
            peso_lista = [w_dict.get(i, 1.0) for i in range(15)]
            current_weight_tensor = torch.tensor(peso_lista, dtype=torch.float32).to(device)
            criterion = nn.CrossEntropyLoss(weight=current_weight_tensor)
        else:
            criterion = nn.CrossEntropyLoss()
            
    elif loss_method == 'focal_loss':
        w_method = 'none' 
        gamma_val = trial.suggest_float('gamma', 0.5, 4.0)
        criterion = FocalLoss(gamma=gamma_val)

    batch_size = trial.suggest_categorical('batch_size', [4096, 8192, 16384])
    n_layers = trial.suggest_int('n_layers', 2, 4)
    n_units = trial.suggest_int('n_units', 256, 1024, step=256)
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)
    activation_name = trial.suggest_categorical('activation', ['leaky_relu', 'gelu'])
    lr = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True)
    epochs = trial.suggest_int('epochs', 50, 100)
    
    model = TabularMLP(input_dim=k_opt, n_layers=n_layers, n_units=n_units, 
                       dropout_rate=dropout_rate, activation_name=activation_name,
                       num_classes=15).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    for epoch in range(epochs):
        model.train()
        permutation = torch.randperm(num_samples).to(device)
        
        for i in range(0, num_samples, batch_size):
            indices = permutation[i:i+batch_size]
            bx, by = X_train_vram[indices], y_train_vram[indices]
            
            optimizer.zero_grad()
            logits = model(bx)
            loss = criterion(logits, by)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        val_logits = model(X_val_vram_final)
        preds = torch.argmax(val_logits, dim=1).cpu().numpy()
        
    f1_mac = f1_score(y_val_cpu, preds, average='macro')
    report = classification_report(y_val_cpu, preds, zero_division=0)
    
    save_confusion_matrix_final(y_val_cpu, preds, trial.number, ds_name, fs_method, w_method, loss_method)
    
    log_msg = f"""Trial {trial.number} - Experimento 6 MLP
Arquitectura: DS={ds_name} | FS={fs_method} | W={w_method} | Loss={loss_method}
Params Completos: {trial.params}

F1 Macro: {f1_mac:.4f}
{report}
"""
    with open(f"{log_folder}/Trial_{trial.number}_Metrics.log", 'w', encoding='utf-8') as f:
        f.write(log_msg)
    
    print(f"Trial {trial.number} | F1: {f1_mac:.4f} | DS: {ds_name} | FS: {fs_method} | W: {w_method} | Loss: {loss_method}")
    
    del model, optimizer, logits, loss, val_logits, criterion, X_train_vram, X_val_vram_final
    gc.collect()
    torch.cuda.empty_cache()
    
    return f1_mac

print("\nIniciando el Experimento 6 de MLP...")
study_final_mlp = optuna.create_study(direction='maximize', study_name="Experimento_6_MLP")

study_final_mlp.optimize(objective_final_mlp, n_trials=100)

print("Resultados Experimento 6")
print(f"Mejor F1 Macro: {study_final_mlp.best_value:.4f}")
print(f"Mejor Configuración:\n{study_final_mlp.best_params}")

with open(f"{log_folder}/Resultados_Experimento_6.txt", 'w', encoding='utf-8') as f:
    f.write("Resumen Experimento 6 MLP:\n")
    f.write(f"Mejor F1 Macro: {study_final_mlp.best_value:.4f}\n")
    f.write(f"Parámetros Ganadores:\n{study_final_mlp.best_params}\n")

[I 2026-04-03 15:24:46,065] A new study created in memory with name: Experimento_6_MLP


Preparando tensor de validación...

Iniciando el Experimento 6 de MLP...
Trial 0 | F1: 0.5255 | DS: none | FS: tree | W: none | Loss: focal_loss


[I 2026-04-03 15:25:36,578] Trial 0 finished with value: 0.525453580912702 and parameters: {'dataset': 'none', 'fs_method': 'tree', 'k_features': 56, 'loss_function': 'focal_loss', 'gamma': 1.9799755333307372, 'batch_size': 16384, 'n_layers': 3, 'n_units': 768, 'dropout_rate': 0.18638617993774242, 'activation': 'gelu', 'lr': 0.0005909534550653529, 'weight_decay': 0.0021348999132060134, 'epochs': 90}. Best is trial 0 with value: 0.525453580912702.
[I 2026-04-03 15:26:09,534] Trial 1 finished with value: 0.6700009365619536 and parameters: {'dataset': 'smote_enn', 'fs_method': 'tree', 'k_features': 61, 'loss_function': 'cross_entropy', 'weight_method': 'none', 'batch_size': 8192, 'n_layers': 2, 'n_units': 512, 'dropout_rate': 0.11181727999058207, 'activation': 'leaky_relu', 'lr': 0.0007009178223256712, 'weight_decay': 0.00040937110017359024, 'epochs': 56}. Best is trial 1 with value: 0.6700009365619536.


Trial 1 | F1: 0.6700 | DS: smote_enn | FS: tree | W: none | Loss: cross_entropy


[I 2026-04-03 15:27:01,727] Trial 2 finished with value: 0.6055783017822948 and parameters: {'dataset': 'none', 'fs_method': 'tree', 'k_features': 49, 'loss_function': 'focal_loss', 'gamma': 2.9195439675516037, 'batch_size': 4096, 'n_layers': 2, 'n_units': 512, 'dropout_rate': 0.12139208081028144, 'activation': 'gelu', 'lr': 0.0031411473962801514, 'weight_decay': 1.0296996474142337e-05, 'epochs': 57}. Best is trial 1 with value: 0.6700009365619536.


...

Trial 97 | F1: 0.7605 | DS: none | FS: none | W: none | Loss: cross_entropy


[I 2026-04-03 16:20:32,632] Trial 97 finished with value: 0.7605028194193061 and parameters: {'dataset': 'none', 'fs_method': 'none', 'loss_function': 'cross_entropy', 'weight_method': 'none', 'batch_size': 16384, 'n_layers': 3, 'n_units': 512, 'dropout_rate': 0.13040295327476936, 'activation': 'gelu', 'lr': 0.00024890387900704085, 'weight_decay': 5.913673467240105e-06, 'epochs': 76}. Best is trial 26 with value: 0.7669376948823251.
[I 2026-04-03 16:20:55,974] Trial 98 finished with value: 0.7508297094392372 and parameters: {'dataset': 'none', 'fs_method': 'none', 'loss_function': 'cross_entropy', 'weight_method': 'none', 'batch_size': 16384, 'n_layers': 3, 'n_units': 512, 'dropout_rate': 0.12824070859758002, 'activation': 'gelu', 'lr': 0.0002614929536051143, 'weight_decay': 6.34279664628515e-06, 'epochs': 76}. Best is trial 26 with value: 0.7669376948823251.


Trial 98 | F1: 0.7508 | DS: none | FS: none | W: none | Loss: cross_entropy
Trial 99 | F1: 0.7250 | DS: none | FS: none | W: none | Loss: cross_entropy


[I 2026-04-03 16:21:25,362] Trial 99 finished with value: 0.7249893868518745 and parameters: {'dataset': 'none', 'fs_method': 'none', 'loss_function': 'cross_entropy', 'weight_method': 'none', 'batch_size': 8192, 'n_layers': 3, 'n_units': 512, 'dropout_rate': 0.11056981490003039, 'activation': 'gelu', 'lr': 0.00023132345800946292, 'weight_decay': 5.730082024037254e-06, 'epochs': 72}. Best is trial 26 with value: 0.7669376948823251.


Resultados Experimento 6
Mejor F1 Macro: 0.7669
Mejor Configuración:
{'dataset': 'none', 'fs_method': 'none', 'loss_function': 'cross_entropy', 'weight_method': 'none', 'batch_size': 16384, 'n_layers': 2, 'n_units': 512, 'dropout_rate': 0.42846027632732114, 'activation': 'gelu', 'lr': 0.0004295534166068324, 'weight_decay': 9.92741862009386e-06, 'epochs': 80}


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import optuna
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import warnings
from sklearn.metrics import f1_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

log_folder = "Logs_MLP_Baseline_7"
os.makedirs(log_folder, exist_ok=True)

def save_confusion_matrix_final(y_true, y_pred, trial_number, ds_name, fs_name, w_name, l_name, phase="Val"):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'CM Final MLP - Trial {trial_number}\nDS: {ds_name} | FS: {fs_name} | W: {w_name} | Loss: {l_name}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig(f'{log_folder}/Trial_{trial_number}_Final_CM.png')
    plt.close()

class TabularMLP(nn.Module):
    def __init__(self, input_dim, n_layers, n_units, dropout_rate, activation_name, num_classes=13):
        super().__init__()
        activations = {'leaky_relu': nn.LeakyReLU(0.01), 'gelu': nn.GELU()}
        activation_layer = activations[activation_name]

        layers = []
        in_f = input_dim
        for i in range(n_layers):
            layers.append(nn.Linear(in_f, n_units))
            layers.append(nn.BatchNorm1d(n_units))
            layers.append(activation_layer)
            layers.append(nn.Dropout(dropout_rate))
            in_f = n_units
        
        layers.append(nn.Linear(in_f, num_classes))
        self.net = nn.Sequential(*layers)
        
    def forward(self, x): 
        return self.net(x)

print("Preparando tensor de validación...")
X_val_raw_cpu = np.array(X_val_imp_grouped, dtype=np.float32)
y_val_vram = torch.tensor(np.array(y_val_grouped, dtype=np.int64)).to(device)
y_val_cpu = y_val_vram.cpu().numpy()

def objective_final_mlp(trial):
    ds_name = trial.suggest_categorical('dataset', ['none', 'smote_enn'])
    x_raw_train_cpu, y_raw_train_cpu = train_datasets_grouped[ds_name]
    total_cols = x_raw_train_cpu.shape[1]
    num_samples = x_raw_train_cpu.shape[0]
    
    y_train_vram = torch.tensor(np.array(y_raw_train_cpu, dtype=np.int64)).to(device)

    fs_method = trial.suggest_categorical('fs_method', ['none', 'tree'])
    if fs_method == 'tree':
        k_opt = trial.suggest_int('k_features', 30, total_cols)
        selector = FeatureSelector(method='tree', k=k_opt)
        X_train_fs = selector.fit_transform(x_raw_train_cpu, y_raw_train_cpu)
        X_val_fs = selector.transform(X_val_raw_cpu)
    else:
        k_opt = total_cols
        X_train_fs = x_raw_train_cpu
        X_val_fs = X_val_raw_cpu

    X_train_vram = torch.tensor(np.array(X_train_fs), dtype=torch.float32).to(device)
    X_val_vram_final = torch.tensor(np.array(X_val_fs), dtype=torch.float32).to(device)

    loss_method = trial.suggest_categorical('loss_function', ['cross_entropy', 'focal_loss'])
    
    if loss_method == 'cross_entropy':
        w_method = trial.suggest_categorical('weight_method', ['none', 'polynomial'])
        
        if w_method == 'polynomial':
            alpha_val = trial.suggest_float('poly_alpha', 0.1, 1.0)
            w_dict = polynomial_class_weight(y_raw_train_cpu, alpha=alpha_val)
            peso_lista = [w_dict.get(i, 1.0) for i in range(13)]
            current_weight_tensor = torch.tensor(peso_lista, dtype=torch.float32).to(device)
            criterion = nn.CrossEntropyLoss(weight=current_weight_tensor)
        else:
            criterion = nn.CrossEntropyLoss()
            
    elif loss_method == 'focal_loss':
        w_method = 'none' 
        gamma_val = trial.suggest_float('gamma', 0.5, 4.0)
        criterion = FocalLoss(gamma=gamma_val)

    batch_size = trial.suggest_categorical('batch_size', [4096, 8192, 16384])
    n_layers = trial.suggest_int('n_layers', 2, 4)
    n_units = trial.suggest_int('n_units', 256, 1024, step=256)
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)
    activation_name = trial.suggest_categorical('activation', ['leaky_relu', 'gelu'])
    lr = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True)
    epochs = trial.suggest_int('epochs', 50, 100)
    
    model = TabularMLP(input_dim=k_opt, n_layers=n_layers, n_units=n_units, 
                       dropout_rate=dropout_rate, activation_name=activation_name,
                       num_classes=13).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    for epoch in range(epochs):
        model.train()
        permutation = torch.randperm(num_samples).to(device)
        
        for i in range(0, num_samples, batch_size):
            indices = permutation[i:i+batch_size]
            bx, by = X_train_vram[indices], y_train_vram[indices]
            
            optimizer.zero_grad()
            logits = model(bx)
            loss = criterion(logits, by)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        val_logits = model(X_val_vram_final)
        preds = torch.argmax(val_logits, dim=1).cpu().numpy()
        
    f1_mac = f1_score(y_val_cpu, preds, average='macro')
    report = classification_report(y_val_cpu, preds, zero_division=0)
    
    save_confusion_matrix_final(y_val_cpu, preds, trial.number, ds_name, fs_method, w_method, loss_method)
    
    log_msg = f"""Trial {trial.number} - Experimento 7 MLP
Arquitectura: DS={ds_name} | FS={fs_method} | W={w_method} | Loss={loss_method}
Params Completos: {trial.params}

F1 Macro: {f1_mac:.4f}
{report}
"""
    with open(f"{log_folder}/Trial_{trial.number}_Metrics.log", 'w', encoding='utf-8') as f:
        f.write(log_msg)
    
    print(f"Trial {trial.number} | F1: {f1_mac:.4f} | DS: {ds_name} | FS: {fs_method} | W: {w_method} | Loss: {loss_method}")
    
    del model, optimizer, logits, loss, val_logits, criterion, X_train_vram, X_val_vram_final
    gc.collect()
    torch.cuda.empty_cache()
    
    return f1_mac

print("\nIniciando el Experimento 7 de MLP...")
study_final_mlp = optuna.create_study(direction='maximize', study_name="Experimento_6_MLP")

study_final_mlp.optimize(objective_final_mlp, n_trials=100)

print("Resultados Experimento 7")
print(f"Mejor F1 Macro: {study_final_mlp.best_value:.4f}")
print(f"Mejor Configuración:\n{study_final_mlp.best_params}")

with open(f"{log_folder}/Resultados_Experimento_7.txt", 'w', encoding='utf-8') as f:
    f.write("Resumen Experimento 7 MLP:\n")
    f.write(f"Mejor F1 Macro: {study_final_mlp.best_value:.4f}\n")
    f.write(f"Parámetros Ganadores:\n{study_final_mlp.best_params}\n")

[I 2026-04-03 16:32:05,562] A new study created in memory with name: Experimento_6_MLP


Preparando tensor de validación...

Iniciando el Experimento 7 de MLP...
Trial 0 | F1: 0.7331 | DS: smote_enn | FS: tree | W: polynomial | Loss: cross_entropy


[I 2026-04-03 16:32:54,945] Trial 0 finished with value: 0.7330996265077812 and parameters: {'dataset': 'smote_enn', 'fs_method': 'tree', 'k_features': 55, 'loss_function': 'cross_entropy', 'weight_method': 'polynomial', 'poly_alpha': 0.6379542636208716, 'batch_size': 8192, 'n_layers': 2, 'n_units': 256, 'dropout_rate': 0.2121627002699821, 'activation': 'gelu', 'lr': 0.002326924083268873, 'weight_decay': 1.8689915749500522e-06, 'epochs': 94}. Best is trial 0 with value: 0.7330996265077812.


Trial 1 | F1: 0.7091 | DS: none | FS: tree | W: none | Loss: focal_loss


[I 2026-04-03 16:33:23,367] Trial 1 finished with value: 0.7091458333263408 and parameters: {'dataset': 'none', 'fs_method': 'tree', 'k_features': 35, 'loss_function': 'focal_loss', 'gamma': 1.9383840814760447, 'batch_size': 16384, 'n_layers': 2, 'n_units': 256, 'dropout_rate': 0.36765422495745703, 'activation': 'gelu', 'lr': 0.0006130538560668526, 'weight_decay': 0.0006439469016278038, 'epochs': 91}. Best is trial 0 with value: 0.7330996265077812.


...

Trial 98 | F1: 0.7074 | DS: smote_enn | FS: none | W: polynomial | Loss: cross_entropy


[I 2026-04-03 18:23:27,344] Trial 98 finished with value: 0.7073573883943314 and parameters: {'dataset': 'smote_enn', 'fs_method': 'none', 'loss_function': 'cross_entropy', 'weight_method': 'polynomial', 'poly_alpha': 0.8273314939506147, 'batch_size': 16384, 'n_layers': 4, 'n_units': 256, 'dropout_rate': 0.17258390274153051, 'activation': 'leaky_relu', 'lr': 0.0005421139515840675, 'weight_decay': 2.301558606492076e-06, 'epochs': 58}. Best is trial 84 with value: 0.9166422488981559.


Trial 99 | F1: 0.7300 | DS: none | FS: tree | W: none | Loss: cross_entropy


[I 2026-04-03 18:24:43,413] Trial 99 finished with value: 0.7300377492770943 and parameters: {'dataset': 'none', 'fs_method': 'tree', 'k_features': 46, 'loss_function': 'cross_entropy', 'weight_method': 'none', 'batch_size': 4096, 'n_layers': 4, 'n_units': 256, 'dropout_rate': 0.11902047376196177, 'activation': 'leaky_relu', 'lr': 0.004481887675524437, 'weight_decay': 1.5536337501103863e-06, 'epochs': 72}. Best is trial 84 with value: 0.9166422488981559.


Resultados Experimento 7
Mejor F1 Macro: 0.9166
Mejor Configuración:
{'dataset': 'smote_enn', 'fs_method': 'none', 'loss_function': 'cross_entropy', 'weight_method': 'none', 'batch_size': 4096, 'n_layers': 2, 'n_units': 256, 'dropout_rate': 0.15366991071861089, 'activation': 'leaky_relu', 'lr': 0.000822912285514681, 'weight_decay': 2.251040105275986e-06, 'epochs': 59}


In [20]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, classification_report, confusion_matrix

os.environ["CUDA_VISIBLE_DEVICES"] = "1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

log_folder_test_mlp = "Logs_Resultados_Test_Final_MLP"
os.makedirs(log_folder_test_mlp, exist_ok=True)

class TabularMLP(nn.Module):
    def __init__(self, input_dim, n_layers, n_units, dropout_rate, activation_name, num_classes=15):
        super().__init__()
        activations = {
            'leaky_relu': nn.LeakyReLU(0.01),
            'gelu': nn.GELU()
        }
        activation_layer = activations[activation_name]

        layers = []
        in_f = input_dim
        for i in range(n_layers):
            layers.append(nn.Linear(in_f, n_units))
            layers.append(nn.BatchNorm1d(n_units))
            layers.append(activation_layer)
            layers.append(nn.Dropout(dropout_rate))
            in_f = n_units
        
        layers.append(nn.Linear(in_f, num_classes))
        self.net = nn.Sequential(*layers)
        
    def forward(self, x): 
        return self.net(x)

print("Preparando tensores en VRAM...")
x_raw_train, y_raw_train = train_datasets['none']

X_train_vram = torch.tensor(np.array(x_raw_train, dtype=np.float32)).to(device)
y_train_vram = torch.tensor(np.array(y_raw_train, dtype=np.int64)).to(device)

X_test_vram = torch.tensor(np.array(X_test, dtype=np.float32)).to(device)
y_test_cpu = np.array(y_test_1d, dtype=np.int64)

input_dimension = X_train_vram.shape[1]
num_samples = X_train_vram.shape[0]

best_params_mlp_baseline = {
    'batch_size': 16384, 
    'n_layers': 2, 
    'n_units': 512, 
    'dropout_rate': 0.18489250809426297, 
    'activation': 'leaky_relu', 
    'lr': 0.00048640065976150775, 
    'weight_decay': 1.3043567069476427e-05, 
    'epochs': 97
}

model = TabularMLP(input_dim=input_dimension, 
                   n_layers=best_params_mlp_baseline['n_layers'], 
                   n_units=best_params_mlp_baseline['n_units'], 
                   dropout_rate=best_params_mlp_baseline['dropout_rate'],
                   activation_name=best_params_mlp_baseline['activation'],
                   num_classes=15).to(device)
                   
optimizer = optim.Adam(model.parameters(), 
                       lr=best_params_mlp_baseline['lr'], 
                       weight_decay=best_params_mlp_baseline['weight_decay'])

print(f"Entrenando MLP Definitiva (Baseline) por {best_params_mlp_baseline['epochs']} épocas...")
batch_size = best_params_mlp_baseline['batch_size']

for epoch in range(best_params_mlp_baseline['epochs']):
    model.train()
    permutation = torch.randperm(num_samples).to(device)
    
    for i in range(0, num_samples, batch_size):
        indices = permutation[i:i+batch_size]
        bx, by = X_train_vram[indices], y_train_vram[indices]
        
        optimizer.zero_grad()
        logits = model(bx)
        loss = F.cross_entropy(logits, by)
        loss.backward()
        optimizer.step()

print("Generando predicciones en el Test Set...")
model.eval()
with torch.no_grad():
    test_logits = model(X_test_vram)
    preds_test_mlp = torch.argmax(test_logits, dim=1).cpu().numpy()

f1_mac_test = f1_score(y_test_cpu, preds_test_mlp, average='macro')
report_test = classification_report(y_test_cpu, preds_test_mlp, zero_division=0)

print(f"\nResultados finales en Test - MLP Baseline")
print(f"F1 Macro Alcanzado: {f1_mac_test:.4f}")
print("\nReporte de Clasificación:")
print(report_test)

cm = confusion_matrix(y_test_cpu, preds_test_mlp)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'CM Final Test Set - MLP Baseline\nF1 Macro: {f1_mac_test:.4f}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig(f'{log_folder_test_mlp}/CM_Test_MLP_Baseline_Exp1.png')
plt.close()

with open(f"{log_folder_test_mlp}/Reporte_Test_MLP_Baseline_Test_1.txt", "w", encoding='utf-8') as f:
    f.write("Evaluacion Final en Test Set - MLP Baseline\n")
    f.write(f"Hiperparámetros de Optuna: {best_params_mlp_baseline}\n\n")
    f.write(f"F1 Macro en Test: {f1_mac_test:.4f}\n\n")
    f.write("Reporte de Clasificación Detallado:\n")
    f.write(report_test)

print(f"\nPrueba finalizada exitosamente. Archivos guardados en '{log_folder_test_mlp}'")

Preparando tensores en VRAM...
Entrenando MLP Definitiva (Baseline) por 97 épocas...
Generando predicciones en el Test Set...

Resultados finales en Test - MLP Baseline
F1 Macro Alcanzado: 0.7558

Reporte de Clasificación:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    307072
           1       0.99      0.33      0.50       295
           2       1.00      1.00      1.00     19204
           3       0.99      0.99      0.99      1544
           4       0.98      1.00      0.99     34661
           5       0.90      0.99      0.94       825
           6       0.99      0.99      0.99       870
           7       1.00      0.99      1.00      1190
           8       1.00      1.00      1.00         2
           9       1.00      0.60      0.75         5
          10       0.99      1.00      1.00     23840
          11       0.94      0.99      0.96       884
          12       0.95      0.09      0.17       226
          13       0

In [21]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, classification_report, confusion_matrix

os.environ["CUDA_VISIBLE_DEVICES"] = "1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

log_folder_test_mlp = "Logs_Resultados_Test_Final_MLP"
os.makedirs(log_folder_test_mlp, exist_ok=True)

class TabularMLP(nn.Module):
    def __init__(self, input_dim, n_layers, n_units, dropout_rate, activation_name, num_classes=15):
        super().__init__()
        activations = {
            'leaky_relu': nn.LeakyReLU(0.01),
            'gelu': nn.GELU()
        }
        activation_layer = activations[activation_name]

        layers = []
        in_f = input_dim
        for i in range(n_layers):
            layers.append(nn.Linear(in_f, n_units))
            layers.append(nn.BatchNorm1d(n_units))
            layers.append(activation_layer)
            layers.append(nn.Dropout(dropout_rate))
            in_f = n_units
        
        layers.append(nn.Linear(in_f, num_classes))
        self.net = nn.Sequential(*layers)
        
    def forward(self, x): 
        return self.net(x)

print("Preparando tensores en VRAM (SMOTE_ENN)...")
x_raw_train, y_raw_train = train_datasets['smote_enn']

X_train_vram = torch.tensor(np.array(x_raw_train, dtype=np.float32)).to(device)
y_train_vram = torch.tensor(np.array(y_raw_train, dtype=np.int64)).to(device)

X_test_vram = torch.tensor(np.array(X_test, dtype=np.float32)).to(device)
y_test_cpu = np.array(y_test_1d, dtype=np.int64)

input_dimension = X_train_vram.shape[1]
num_samples = X_train_vram.shape[0]

best_params_mlp_exp2 = {
    'batch_size': 4096, 
    'n_layers': 3, 
    'n_units': 768, 
    'dropout_rate': 0.29647145408290937, 
    'activation': 'leaky_relu', 
    'lr': 0.00016332182321991082, 
    'weight_decay': 8.033054547048967e-06, 
    'epochs': 78
}

model = TabularMLP(input_dim=input_dimension, 
                   n_layers=best_params_mlp_exp2['n_layers'], 
                   n_units=best_params_mlp_exp2['n_units'], 
                   dropout_rate=best_params_mlp_exp2['dropout_rate'],
                   activation_name=best_params_mlp_exp2['activation'],
                   num_classes=15).to(device)
                   
optimizer = optim.Adam(model.parameters(), 
                       lr=best_params_mlp_exp2['lr'], 
                       weight_decay=best_params_mlp_exp2['weight_decay'])

print(f"Entrenando MLP Definitiva (Experimento 2 - SMOTE_ENN) por {best_params_mlp_exp2['epochs']} épocas...")
batch_size = best_params_mlp_exp2['batch_size']

for epoch in range(best_params_mlp_exp2['epochs']):
    model.train()
    permutation = torch.randperm(num_samples).to(device)
    
    for i in range(0, num_samples, batch_size):
        indices = permutation[i:i+batch_size]
        bx, by = X_train_vram[indices], y_train_vram[indices]
        
        optimizer.zero_grad()
        logits = model(bx)
        loss = F.cross_entropy(logits, by)
        loss.backward()
        optimizer.step()

print("Generando predicciones en el Test Set...")
model.eval()
with torch.no_grad():
    test_logits = model(X_test_vram)
    preds_test_mlp = torch.argmax(test_logits, dim=1).cpu().numpy()

f1_mac_test = f1_score(y_test_cpu, preds_test_mlp, average='macro')
report_test = classification_report(y_test_cpu, preds_test_mlp, zero_division=0)

print(f"\nResultados finales en Test - MLP Experimento 2 (SMOTE_ENN)")
print(f"F1 Macro Alcanzado: {f1_mac_test:.4f}")
print("\nReporte de Clasificación:")
print(report_test)

cm = confusion_matrix(y_test_cpu, preds_test_mlp)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'CM Final Test Set - MLP Experimento 2 (SMOTE_ENN)\nF1 Macro: {f1_mac_test:.4f}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig(f'{log_folder_test_mlp}/CM_Test_MLP_Baseline_Exp2.png')
plt.close()

with open(f"{log_folder_test_mlp}/Reporte_Test_MLP_Test_2.txt", "w", encoding='utf-8') as f:
    f.write("Evaluacion Final en Test Set - MLP Experimento 2 (SMOTE_ENN)\n")
    f.write(f"Hiperparámetros de Optuna: {best_params_mlp_exp2}\n\n")
    f.write(f"F1 Macro en Test: {f1_mac_test:.4f}\n\n")
    f.write("Reporte de Clasificación Detallado:\n")
    f.write(report_test)

print(f"\nPrueba finalizada exitosamente. Archivos guardados en '{log_folder_test_mlp}'")

Preparando tensores en VRAM (SMOTE_ENN)...
Entrenando MLP Definitiva (Experimento 2 - SMOTE_ENN) por 78 épocas...
Generando predicciones en el Test Set...

Resultados finales en Test - MLP Experimento 2 (SMOTE_ENN)
F1 Macro Alcanzado: 0.7329

Reporte de Clasificación:
              precision    recall  f1-score   support

           0       1.00      0.99      1.00    307072
           1       0.22      0.83      0.35       295
           2       1.00      1.00      1.00     19204
           3       0.97      1.00      0.98      1544
           4       0.98      1.00      0.99     34661
           5       0.90      0.99      0.95       825
           6       0.98      1.00      0.99       870
           7       1.00      1.00      1.00      1190
           8       0.67      1.00      0.80         2
           9       0.75      0.60      0.67         5
          10       0.99      1.00      1.00     23840
          11       0.94      0.99      0.97       884
          12       0.16     

In [22]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, classification_report, confusion_matrix

os.environ["CUDA_VISIBLE_DEVICES"] = "1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

log_folder_test_mlp = "Logs_Resultados_Test_Final_MLP"
os.makedirs(log_folder_test_mlp, exist_ok=True)

class TabularMLP(nn.Module):
    def __init__(self, input_dim, n_layers, n_units, dropout_rate, activation_name, num_classes=15):
        super().__init__()
        activations = {
            'leaky_relu': nn.LeakyReLU(0.01),
            'gelu': nn.GELU()
        }
        activation_layer = activations[activation_name]

        layers = []
        in_f = input_dim
        for i in range(n_layers):
            layers.append(nn.Linear(in_f, n_units))
            layers.append(nn.BatchNorm1d(n_units))
            layers.append(activation_layer)
            layers.append(nn.Dropout(dropout_rate))
            in_f = n_units
        
        layers.append(nn.Linear(in_f, num_classes))
        self.net = nn.Sequential(*layers)
        
    def forward(self, x): 
        return self.net(x)

def polynomial_class_weight(y, alpha=0.25):
    classes, counts = np.unique(y, return_counts=True)
    weights = 1.0 / np.power(counts, alpha)
    weights = weights / np.mean(weights)
    return dict(zip(classes, weights))

print("Preparando tensores en VRAM (Dataset Original para Pesos)...")
x_raw_train, y_raw_train = train_datasets['none']

X_train_vram = torch.tensor(np.array(x_raw_train, dtype=np.float32)).to(device)
y_train_vram = torch.tensor(np.array(y_raw_train, dtype=np.int64)).to(device)

X_test_vram = torch.tensor(np.array(X_test, dtype=np.float32)).to(device)
y_test_cpu = np.array(y_test_1d, dtype=np.int64)

input_dimension = X_train_vram.shape[1]
num_samples = X_train_vram.shape[0]

best_params_mlp_exp3 = {
    'batch_size': 16384, 
    'n_layers': 3, 
    'n_units': 512, 
    'dropout_rate': 0.2630967973253173, 
    'activation': 'leaky_relu', 
    'lr': 0.0002269343077526274, 
    'weight_decay': 0.0003556939051918942, 
    'epochs': 61, 
    'poly_alpha': 0.34183481330155585
}

print("Calculando el tensor de pesos polinomiales...")
w_dict = polynomial_class_weight(y_raw_train, alpha=best_params_mlp_exp3['poly_alpha'])
peso_lista = [w_dict.get(i, 1.0) for i in range(15)]
current_weight_tensor = torch.tensor(peso_lista, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=current_weight_tensor)

model = TabularMLP(input_dim=input_dimension, 
                   n_layers=best_params_mlp_exp3['n_layers'], 
                   n_units=best_params_mlp_exp3['n_units'], 
                   dropout_rate=best_params_mlp_exp3['dropout_rate'],
                   activation_name=best_params_mlp_exp3['activation'],
                   num_classes=15).to(device)
                   
optimizer = optim.Adam(model.parameters(), 
                       lr=best_params_mlp_exp3['lr'], 
                       weight_decay=best_params_mlp_exp3['weight_decay'])

print(f"Entrenando MLP Definitiva (Experimento 3 - POLYNOMIAL) por {best_params_mlp_exp3['epochs']} épocas...")
batch_size = best_params_mlp_exp3['batch_size']

for epoch in range(best_params_mlp_exp3['epochs']):
    model.train()
    permutation = torch.randperm(num_samples).to(device)
    
    for i in range(0, num_samples, batch_size):
        indices = permutation[i:i+batch_size]
        bx, by = X_train_vram[indices], y_train_vram[indices]
        
        optimizer.zero_grad()
        logits = model(bx)
        loss = criterion(logits, by) 
        loss.backward()
        optimizer.step()

print("Generando predicciones en el Test Set...")
model.eval()
with torch.no_grad():
    test_logits = model(X_test_vram)
    preds_test_mlp = torch.argmax(test_logits, dim=1).cpu().numpy()

f1_mac_test = f1_score(y_test_cpu, preds_test_mlp, average='macro')
report_test = classification_report(y_test_cpu, preds_test_mlp, zero_division=0)

print(f"\nResultados finales en Test - MLP Experimento 3 (POLYNOMIAL)")
print(f"F1 Macro Alcanzado: {f1_mac_test:.4f}")
print("\nReporte de Clasificación:")
print(report_test)

cm = confusion_matrix(y_test_cpu, preds_test_mlp)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'CM Final Test Set - MLP Experimento 3 (POLYNOMIAL)\nF1 Macro: {f1_mac_test:.4f}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig(f'{log_folder_test_mlp}/CM_Test_MLP_Baseline_Exp3.png')
plt.close()

with open(f"{log_folder_test_mlp}/Reporte_Test_MLP_Test_3.txt", "w", encoding='utf-8') as f:
    f.write("Evaluacion Final en Test Set - MLP Experimento 3 (POLYNOMIAL)\n")
    f.write(f"Hiperparámetros de Optuna: {best_params_mlp_exp3}\n\n")
    f.write(f"F1 Macro en Test: {f1_mac_test:.4f}\n\n")
    f.write("Reporte de Clasificación Detallado:\n")
    f.write(report_test)

print(f"\nPrueba finalizada exitosamente. Archivos guardados en '{log_folder_test_mlp}'")

Preparando tensores en VRAM (Dataset Original para Pesos)...
Calculando el tensor de pesos polinomiales...
Entrenando MLP Definitiva (Experimento 3 - POLYNOMIAL) por 61 épocas...
Generando predicciones en el Test Set...

Resultados finales en Test - MLP Experimento 3 (POLYNOMIAL)
F1 Macro Alcanzado: 0.7486

Reporte de Clasificación:
              precision    recall  f1-score   support

           0       1.00      0.99      0.99    307072
           1       0.56      0.58      0.57       295
           2       1.00      1.00      1.00     19204
           3       0.97      0.99      0.98      1544
           4       0.97      1.00      0.98     34661
           5       0.89      0.99      0.94       825
           6       0.96      0.99      0.97       870
           7       1.00      0.99      0.99      1190
           8       0.67      1.00      0.80         2
           9       0.80      0.80      0.80         5
          10       0.99      1.00      1.00     23840
          11    

In [23]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, classification_report, confusion_matrix

os.environ["CUDA_VISIBLE_DEVICES"] = "1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

log_folder_test_mlp = "Logs_Resultados_Test_Final_MLP"
os.makedirs(log_folder_test_mlp, exist_ok=True)

class TabularMLP(nn.Module):
    def __init__(self, input_dim, n_layers, n_units, dropout_rate, activation_name, num_classes=15):
        super().__init__()
        activations = {
            'leaky_relu': nn.LeakyReLU(0.01),
            'gelu': nn.GELU()
        }
        activation_layer = activations[activation_name]

        layers = []
        in_f = input_dim
        for i in range(n_layers):
            layers.append(nn.Linear(in_f, n_units))
            layers.append(nn.BatchNorm1d(n_units))
            layers.append(activation_layer)
            layers.append(nn.Dropout(dropout_rate))
            in_f = n_units
        
        layers.append(nn.Linear(in_f, num_classes))
        self.net = nn.Sequential(*layers)
        
    def forward(self, x): 
        return self.net(x)

print("Preparando datos crudos...")
x_raw_train, y_raw_train = train_datasets['none']

best_params_mlp_exp4 = {
    'k_features': 64, 
    'batch_size': 16384, 
    'n_layers': 2, 
    'n_units': 512, 
    'dropout_rate': 0.4714157544032792, 
    'activation': 'gelu', 
    'lr': 0.00039110752343778766, 
    'weight_decay': 2.8115854270476626e-06, 
    'epochs': 95
}

print(f"Aplicando Feature Selection (Método: TREE, k={best_params_mlp_exp4['k_features']})...")
selector = FeatureSelector(method='tree', k=best_params_mlp_exp4['k_features'])
X_train_fs = selector.fit_transform(x_raw_train, y_raw_train)
X_test_fs = selector.transform(X_test)

print("Subiendo tensores filtrados a VRAM...")
X_train_vram = torch.tensor(np.array(X_train_fs, dtype=np.float32)).to(device)
y_train_vram = torch.tensor(np.array(y_raw_train, dtype=np.int64)).to(device)

X_test_vram = torch.tensor(np.array(X_test_fs, dtype=np.float32)).to(device)
y_test_cpu = np.array(y_test_1d, dtype=np.int64)

input_dimension = X_train_vram.shape[1]
num_samples = X_train_vram.shape[0]

model = TabularMLP(input_dim=input_dimension, 
                   n_layers=best_params_mlp_exp4['n_layers'], 
                   n_units=best_params_mlp_exp4['n_units'], 
                   dropout_rate=best_params_mlp_exp4['dropout_rate'],
                   activation_name=best_params_mlp_exp4['activation'],
                   num_classes=15).to(device)
                   
optimizer = optim.Adam(model.parameters(), 
                       lr=best_params_mlp_exp4['lr'], 
                       weight_decay=best_params_mlp_exp4['weight_decay'])

print(f"Entrenando MLP Definitiva (Experimento 4 - FS TREE) por {best_params_mlp_exp4['epochs']} épocas...")
batch_size = best_params_mlp_exp4['batch_size']

for epoch in range(best_params_mlp_exp4['epochs']):
    model.train()
    permutation = torch.randperm(num_samples).to(device)
    
    for i in range(0, num_samples, batch_size):
        indices = permutation[i:i+batch_size]
        bx, by = X_train_vram[indices], y_train_vram[indices]
        
        optimizer.zero_grad()
        logits = model(bx)
        loss = F.cross_entropy(logits, by)
        loss.backward()
        optimizer.step()

print("Generando predicciones en el Test Set filtrado...")
model.eval()
with torch.no_grad():
    test_logits = model(X_test_vram)
    preds_test_mlp = torch.argmax(test_logits, dim=1).cpu().numpy()

f1_mac_test = f1_score(y_test_cpu, preds_test_mlp, average='macro')
report_test = classification_report(y_test_cpu, preds_test_mlp, zero_division=0)

print(f"\nResultados finales en Test - MLP Experimento 4 (FS TREE)")
print(f"F1 Macro Alcanzado: {f1_mac_test:.4f}")
print("\nReporte de Clasificación:")
print(report_test)

cm = confusion_matrix(y_test_cpu, preds_test_mlp)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'CM Final Test Set - MLP Experimento 4 (FS TREE)\nF1 Macro: {f1_mac_test:.4f}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig(f'{log_folder_test_mlp}/CM_Test_MLP_Baseline_Exp4.png')
plt.close()

with open(f"{log_folder_test_mlp}/Reporte_Test_MLP_Test_4.txt", "w", encoding='utf-8') as f:
    f.write("Evaluacion Final en Test Set - MLP Experimento 4 (FS TREE)\n")
    f.write(f"k_features seleccionadas: {best_params_mlp_exp4['k_features']}\n")
    f.write(f"Hiperparámetros de Optuna: {best_params_mlp_exp4}\n\n")
    f.write(f"F1 Macro en Test: {f1_mac_test:.4f}\n\n")
    f.write("Reporte de Clasificación Detallado:\n")
    f.write(report_test)

print(f"\nPrueba finalizada exitosamente. Archivos guardados en '{log_folder_test_mlp}'")

Preparando datos crudos...
Aplicando Feature Selection (Método: TREE, k=64)...
Subiendo tensores filtrados a VRAM...
Entrenando MLP Definitiva (Experimento 4 - FS TREE) por 95 épocas...
Generando predicciones en el Test Set filtrado...

Resultados finales en Test - MLP Experimento 4 (FS TREE)
F1 Macro Alcanzado: 0.7544

Reporte de Clasificación:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    307072
           1       0.95      0.34      0.50       295
           2       1.00      1.00      1.00     19204
           3       0.98      0.99      0.99      1544
           4       0.98      1.00      0.99     34661
           5       0.90      0.98      0.94       825
           6       0.97      0.99      0.98       870
           7       1.00      0.99      0.99      1190
           8       1.00      1.00      1.00         2
           9       1.00      0.60      0.75         5
          10       0.99      1.00      1.00     23840
   

In [24]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, classification_report, confusion_matrix

os.environ["CUDA_VISIBLE_DEVICES"] = "1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

log_folder_test_mlp = "Logs_Resultados_Test_Final_MLP"
os.makedirs(log_folder_test_mlp, exist_ok=True)

class TabularMLP(nn.Module):
    def __init__(self, input_dim, n_layers, n_units, dropout_rate, activation_name, num_classes=15):
        super().__init__()
        activations = {
            'leaky_relu': nn.LeakyReLU(0.01),
            'gelu': nn.GELU()
        }
        activation_layer = activations[activation_name]

        layers = []
        in_f = input_dim
        for i in range(n_layers):
            layers.append(nn.Linear(in_f, n_units))
            layers.append(nn.BatchNorm1d(n_units))
            layers.append(activation_layer)
            layers.append(nn.Dropout(dropout_rate))
            in_f = n_units
        
        layers.append(nn.Linear(in_f, num_classes))
        self.net = nn.Sequential(*layers)
        
    def forward(self, x): 
        return self.net(x)

print("Preparando tensores en VRAM (Dataset Original)...")
x_raw_train, y_raw_train = train_datasets['none']

X_train_vram = torch.tensor(np.array(x_raw_train, dtype=np.float32)).to(device)
y_train_vram = torch.tensor(np.array(y_raw_train, dtype=np.int64)).to(device)

X_test_vram = torch.tensor(np.array(X_test, dtype=np.float32)).to(device)
y_test_cpu = np.array(y_test_1d, dtype=np.int64)

input_dimension = X_train_vram.shape[1]
num_samples = X_train_vram.shape[0]

best_params_mlp_exp5 = {
    'batch_size': 16384, 
    'n_layers': 3, 
    'n_units': 1024, 
    'dropout_rate': 0.341106268616704, 
    'activation': 'leaky_relu', 
    'lr': 0.000502104608652804, 
    'weight_decay': 1.1217280027951424e-06, 
    'epochs': 91, 
    'gamma': 3.5114736857231303
}

model = TabularMLP(input_dim=input_dimension, 
                   n_layers=best_params_mlp_exp5['n_layers'], 
                   n_units=best_params_mlp_exp5['n_units'], 
                   dropout_rate=best_params_mlp_exp5['dropout_rate'],
                   activation_name=best_params_mlp_exp5['activation'],
                   num_classes=15).to(device)
                   
optimizer = optim.Adam(model.parameters(), 
                       lr=best_params_mlp_exp5['lr'], 
                       weight_decay=best_params_mlp_exp5['weight_decay'])

criterion = FocalLoss(gamma=best_params_mlp_exp5['gamma'])

print(f"Entrenando MLP Definitiva (Experimento 5 - FOCAL LOSS) por {best_params_mlp_exp5['epochs']} épocas...")
batch_size = best_params_mlp_exp5['batch_size']

for epoch in range(best_params_mlp_exp5['epochs']):
    model.train()
    permutation = torch.randperm(num_samples).to(device)
    
    for i in range(0, num_samples, batch_size):
        indices = permutation[i:i+batch_size]
        bx, by = X_train_vram[indices], y_train_vram[indices]
        
        optimizer.zero_grad()
        logits = model(bx)
        loss = criterion(logits, by) 
        loss.backward()
        optimizer.step()

print("Generando predicciones en el Test Set...")
model.eval()
with torch.no_grad():
    test_logits = model(X_test_vram)
    preds_test_mlp = torch.argmax(test_logits, dim=1).cpu().numpy()

f1_mac_test = f1_score(y_test_cpu, preds_test_mlp, average='macro')
report_test = classification_report(y_test_cpu, preds_test_mlp, zero_division=0)

print(f"\nResultados finales en Test - MLP Experimento 5 (FOCAL LOSS)")
print(f"F1 Macro Alcanzado: {f1_mac_test:.4f}")
print("\nReporte de Clasificación:")
print(report_test)

cm = confusion_matrix(y_test_cpu, preds_test_mlp)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'CM Final Test Set - MLP Experimento 5 (FOCAL LOSS)\nF1 Macro: {f1_mac_test:.4f}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig(f'{log_folder_test_mlp}/CM_Test_MLP_Baseline_Exp5.png')
plt.close()

with open(f"{log_folder_test_mlp}/Reporte_Test_MLP_Test_5.txt", "w", encoding='utf-8') as f:
    f.write("Evaluacion Final en Test Set - MLP Experimento 5 (FOCAL LOSS)\n")
    f.write(f"Gamma optimizado: {best_params_mlp_exp5['gamma']}\n")
    f.write(f"Hiperparámetros de Optuna: {best_params_mlp_exp5}\n\n")
    f.write(f"F1 Macro en Test: {f1_mac_test:.4f}\n\n")
    f.write("Reporte de Clasificación Detallado:\n")
    f.write(report_test)

print(f"\nPrueba finalizada exitosamente. Archivos guardados en '{log_folder_test_mlp}'")

Preparando tensores en VRAM (Dataset Original)...
Entrenando MLP Definitiva (Experimento 5 - FOCAL LOSS) por 91 épocas...
Generando predicciones en el Test Set...

Resultados finales en Test - MLP Experimento 5 (FOCAL LOSS)
F1 Macro Alcanzado: 0.7248

Reporte de Clasificación:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    307072
           1       0.98      0.30      0.46       295
           2       1.00      1.00      1.00     19204
           3       0.99      0.99      0.99      1544
           4       0.98      1.00      0.99     34661
           5       0.90      0.98      0.94       825
           6       0.97      0.99      0.98       870
           7       1.00      1.00      1.00      1190
           8       1.00      1.00      1.00         2
           9       1.00      0.20      0.33         5
          10       0.99      1.00      1.00     23840
          11       0.95      0.99      0.97       884
          12       

In [25]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, classification_report, confusion_matrix

os.environ["CUDA_VISIBLE_DEVICES"] = "1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

log_folder_test_mlp = "Logs_Resultados_Test_Final_MLP"
os.makedirs(log_folder_test_mlp, exist_ok=True)

class TabularMLP(nn.Module):
    def __init__(self, input_dim, n_layers, n_units, dropout_rate, activation_name, num_classes=15):
        super().__init__()
        activations = {
            'leaky_relu': nn.LeakyReLU(0.01),
            'gelu': nn.GELU()
        }
        activation_layer = activations[activation_name]

        layers = []
        in_f = input_dim
        for i in range(n_layers):
            layers.append(nn.Linear(in_f, n_units))
            layers.append(nn.BatchNorm1d(n_units))
            layers.append(activation_layer)
            layers.append(nn.Dropout(dropout_rate))
            in_f = n_units
        
        layers.append(nn.Linear(in_f, num_classes))
        self.net = nn.Sequential(*layers)
        
    def forward(self, x): 
        return self.net(x)

print("Preparando tensores en VRAM (Dataset Original para Arquitectura Final)...")
x_raw_train, y_raw_train = train_datasets['none']

X_train_vram = torch.tensor(np.array(x_raw_train, dtype=np.float32)).to(device)
y_train_vram = torch.tensor(np.array(y_raw_train, dtype=np.int64)).to(device)

X_test_vram = torch.tensor(np.array(X_test, dtype=np.float32)).to(device)
y_test_cpu = np.array(y_test_1d, dtype=np.int64)

input_dimension = X_train_vram.shape[1]
num_samples = X_train_vram.shape[0]

best_params_mlp_exp6 = {
    'batch_size': 16384, 
    'n_layers': 2, 
    'n_units': 512, 
    'dropout_rate': 0.42846027632732114, 
    'activation': 'gelu', 
    'lr': 0.0004295534166068324, 
    'weight_decay': 9.92741862009386e-06, 
    'epochs': 80
}

model = TabularMLP(input_dim=input_dimension, 
                   n_layers=best_params_mlp_exp6['n_layers'], 
                   n_units=best_params_mlp_exp6['n_units'], 
                   dropout_rate=best_params_mlp_exp6['dropout_rate'],
                   activation_name=best_params_mlp_exp6['activation'],
                   num_classes=15).to(device)
                   
optimizer = optim.Adam(model.parameters(), 
                       lr=best_params_mlp_exp6['lr'], 
                       weight_decay=best_params_mlp_exp6['weight_decay'])

print(f"Entrenando MLP Definitiva (Experimento 6 - Arquitectura Final) por {best_params_mlp_exp6['epochs']} épocas...")
batch_size = best_params_mlp_exp6['batch_size']

for epoch in range(best_params_mlp_exp6['epochs']):
    model.train()
    permutation = torch.randperm(num_samples).to(device)
    
    for i in range(0, num_samples, batch_size):
        indices = permutation[i:i+batch_size]
        bx, by = X_train_vram[indices], y_train_vram[indices]
        
        optimizer.zero_grad()
        logits = model(bx)
        loss = F.cross_entropy(logits, by) 
        loss.backward()
        optimizer.step()

print("Generando predicciones en el Test Set...")
model.eval()
with torch.no_grad():
    test_logits = model(X_test_vram)
    preds_test_mlp = torch.argmax(test_logits, dim=1).cpu().numpy()

f1_mac_test = f1_score(y_test_cpu, preds_test_mlp, average='macro')
report_test = classification_report(y_test_cpu, preds_test_mlp, zero_division=0)

print(f"\nResultados finales en Test - MLP Experimento 6 (Arquitectura Final)")
print(f"F1 Macro Alcanzado: {f1_mac_test:.4f}")
print("\nReporte de Clasificación:")
print(report_test)

cm = confusion_matrix(y_test_cpu, preds_test_mlp)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'CM Final Test Set - MLP Experimento 6 (Arquitectura Final)\nF1 Macro: {f1_mac_test:.4f}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig(f'{log_folder_test_mlp}/CM_Test_MLP_Baseline_Exp6.png')
plt.close()

with open(f"{log_folder_test_mlp}/Reporte_Test_MLP_Test_6.txt", "w", encoding='utf-8') as f:
    f.write("Evaluacion Final en Test Set - MLP Experimento 6 (Arquitectura Final)\n")
    f.write(f"Configuración: Dataset='none', FS='none', Loss='cross_entropy', Weight='none'\n")
    f.write(f"Hiperparámetros de Optuna: {best_params_mlp_exp6}\n\n")
    f.write(f"F1 Macro en Test: {f1_mac_test:.4f}\n\n")
    f.write("Reporte de Clasificación Detallado:\n")
    f.write(report_test)

print(f"\nPrueba finalizada exitosamente. Archivos guardados en '{log_folder_test_mlp}'")

Preparando tensores en VRAM (Dataset Original para Arquitectura Final)...
Entrenando MLP Definitiva (Experimento 6 - Arquitectura Final) por 80 épocas...
Generando predicciones en el Test Set...

Resultados finales en Test - MLP Experimento 6 (Arquitectura Final)
F1 Macro Alcanzado: 0.7682

Reporte de Clasificación:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    307072
           1       0.59      0.58      0.59       295
           2       1.00      1.00      1.00     19204
           3       0.99      0.99      0.99      1544
           4       0.97      1.00      0.99     34661
           5       0.90      0.98      0.94       825
           6       0.98      0.95      0.96       870
           7       0.99      0.99      0.99      1190
           8       1.00      1.00      1.00         2
           9       1.00      0.80      0.89         5
          10       0.99      1.00      1.00     23840
          11       0.94      0.98

In [26]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, classification_report, confusion_matrix

os.environ["CUDA_VISIBLE_DEVICES"] = "1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

log_folder_test_mlp = "Logs_Resultados_Test_Final_MLP"
os.makedirs(log_folder_test_mlp, exist_ok=True)

class TabularMLP(nn.Module):
    def __init__(self, input_dim, n_layers, n_units, dropout_rate, activation_name, num_classes=13):
        super().__init__()
        activations = {
            'leaky_relu': nn.LeakyReLU(0.01),
            'gelu': nn.GELU()
        }
        activation_layer = activations[activation_name]

        layers = []
        in_f = input_dim
        for i in range(n_layers):
            layers.append(nn.Linear(in_f, n_units))
            layers.append(nn.BatchNorm1d(n_units))
            layers.append(activation_layer)
            layers.append(nn.Dropout(dropout_rate))
            in_f = n_units
        
        layers.append(nn.Linear(in_f, num_classes))
        self.net = nn.Sequential(*layers)
        
    def forward(self, x): 
        return self.net(x)

print("Preparando tensores en VRAM (Datos Agrupados: SMOTE_ENN)...")
x_train_grp, y_train_grp = train_datasets_grouped['smote_enn']

X_train_vram = torch.tensor(np.array(x_train_grp, dtype=np.float32)).to(device)
y_train_vram = torch.tensor(np.array(y_train_grp, dtype=np.int64)).to(device)

X_test_vram = torch.tensor(np.array(X_test_imp_grouped, dtype=np.float32)).to(device)

y_test_grp_1d = y_test_grouped.to_numpy().ravel() if isinstance(y_test_grouped, pd.DataFrame) else y_test_grouped.to_numpy()
y_test_cpu = np.array(y_test_grp_1d, dtype=np.int64)

input_dimension = X_train_vram.shape[1]
num_samples = X_train_vram.shape[0]

best_params_mlp_exp7 = {
    'batch_size': 4096, 
    'n_layers': 2, 
    'n_units': 256, 
    'dropout_rate': 0.15366991071861089, 
    'activation': 'leaky_relu', 
    'lr': 0.000822912285514681, 
    'weight_decay': 2.251040105275986e-06, 
    'epochs': 59
}

model = TabularMLP(input_dim=input_dimension, 
                   n_layers=best_params_mlp_exp7['n_layers'], 
                   n_units=best_params_mlp_exp7['n_units'], 
                   dropout_rate=best_params_mlp_exp7['dropout_rate'],
                   activation_name=best_params_mlp_exp7['activation'],
                   num_classes=13).to(device)
                   
optimizer = optim.Adam(model.parameters(), 
                       lr=best_params_mlp_exp7['lr'], 
                       weight_decay=best_params_mlp_exp7['weight_decay'])

print(f"Entrenando MLP Definitiva (Experimento 7 - Datos Agrupados) por {best_params_mlp_exp7['epochs']} épocas...")
batch_size = best_params_mlp_exp7['batch_size']

for epoch in range(best_params_mlp_exp7['epochs']):
    model.train()
    permutation = torch.randperm(num_samples).to(device)
    
    for i in range(0, num_samples, batch_size):
        indices = permutation[i:i+batch_size]
        bx, by = X_train_vram[indices], y_train_vram[indices]
        
        optimizer.zero_grad()
        logits = model(bx)
        loss = F.cross_entropy(logits, by)
        loss.backward()
        optimizer.step()

print("Generando predicciones en el Test Set agrupado...")
model.eval()
with torch.no_grad():
    test_logits = model(X_test_vram)
    preds_test_mlp = torch.argmax(test_logits, dim=1).cpu().numpy()

f1_mac_test = f1_score(y_test_cpu, preds_test_mlp, average='macro')
report_test = classification_report(y_test_cpu, preds_test_mlp, zero_division=0)

print(f"\nResultados finales en Test - MLP Experimento 7 (Datos Agrupados)")
print(f"F1 Macro Alcanzado: {f1_mac_test:.4f}")
print("\nReporte de Clasificación:")
print(report_test)

cm = confusion_matrix(y_test_cpu, preds_test_mlp)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'CM Final Test Set - MLP Experimento 7 (Grouped)\nF1 Macro: {f1_mac_test:.4f}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig(f'{log_folder_test_mlp}/CM_Test_MLP_Baseline_Exp7.png')
plt.close()

with open(f"{log_folder_test_mlp}/Reporte_Test_MLP_Test_7.txt", "w", encoding='utf-8') as f:
    f.write("Evaluacion Final en Test Set - MLP Experimento 7 (Datos Agrupados)\n")
    f.write(f"Configuración: Dataset='smote_enn', FS='none', Loss='cross_entropy', Weight='none', num_classes=13\n")
    f.write(f"Hiperparámetros de Optuna: {best_params_mlp_exp7}\n\n")
    f.write(f"F1 Macro en Test: {f1_mac_test:.4f}\n\n")
    f.write("Reporte de Clasificación Detallado:\n")
    f.write(report_test)

print(f"\nPrueba finalizada exitosamente. Archivos guardados en '{log_folder_test_mlp}'")

Preparando tensores en VRAM (Datos Agrupados: SMOTE_ENN)...
Entrenando MLP Definitiva (Experimento 7 - Datos Agrupados) por 59 épocas...
Generando predicciones en el Test Set agrupado...

Resultados finales en Test - MLP Experimento 7 (Datos Agrupados)
F1 Macro Alcanzado: 0.7642

Reporte de Clasificación:
              precision    recall  f1-score   support

           0       1.00      0.99      1.00    307072
           1       0.20      0.86      0.32       295
           2       1.00      1.00      1.00     19204
           3       0.97      1.00      0.98      1544
           4       0.99      1.00      1.00     34661
           5       0.89      0.99      0.93       825
           6       0.99      0.99      0.99       870
           7       0.99      1.00      0.99      1190
           8       0.00      0.00      0.00         2
           9       0.50      0.40      0.44         5
          10       0.99      1.00      1.00     23840
          11       0.95      0.99      0.97 